# Multi-Label Hate Speech and Abusive Language Detection in Indonesian Tweets

This notebook contains the whole study, from the data and preprocessing to the methods, experiments, evaluation, statistical tests, error analysis, follow-up experiments and cross-corpus evaluation. The dataset is the Indonesian Twitter corpus of Ibrohim and Budi (2019), with 12 hate speech and abusive language labels.

The study answers four research questions.

RQ1. How do four algorithm families, namely Bayesian, support vector machine, tree ensemble and neural network, perform on the 12 labels when they are evaluated with metrics suited to imbalanced data?

RQ2. Do class weighting, MLSMOTE and per-label threshold tuning improve performance, especially on the minority labels?

RQ3. Are the differences between the algorithms statistically significant?

RQ4. How well do the models generalise to another corpus, IndoToxic2024?

The first part of the notebook defines every function, grouped by topic. The second part runs the study: exploratory analysis, preprocessing, folds, MLSMOTE inspection, the classical model grid, the IndoBERTweet baseline, evaluation, error analysis, follow-up experiments and the cross-corpus evaluation.

To run the study, run all cells from the top. The expensive parts resume where they stopped. Runs already recorded in results/experiments.csv or results/experiments_ext.csv are skipped, so the notebook can be rerun safely. XGBoost, the MLP and the transformers use CUDA when it is available.

Leakage rule. TF-IDF, resampling, hyperparameter selection, threshold selection, epoch selection and ensemble thresholds use only the training part of each outer fold. The outer test fold is never used to choose anything.

# Naming Conventions

The results files store short codes. This notebook and the paper use the standard terms below.

Algorithms. cnb is Complement Naive Bayes, svm is the linear support vector machine LinearSVC, xgb is XGBoost, mlp is the multilayer perceptron and rf is the random forest.

Transformers. bert is IndoBERTweet fine-tuned with the baseline protocol. tf is a transformer fine-tuned with the revised protocol, and ens is an ensemble that averages the probabilities of several tf models.

Multi-label strategies. br is Binary Relevance, cc is Classifier Chains and lp is Label Powerset. mo means one multi-label output layer, which is how the transformers predict all labels at once.

Imbalance treatment. The condition column holds none, weight, mlsmote, mlsmote_raw or mlros. These mean no treatment, class weighting, MLSMOTE with hierarchy repair, the original MLSMOTE without repair, and ML-ROS. Derived tables call this column imbalance_treatment.

Tuning. fixed means the default decision threshold. tuned means a small hyperparameter search plus one decision threshold per label, both chosen on inner validation data.

Features. word is word unigram and bigram TF-IDF, char is character 3 to 5 gram TF-IDF, and wordchar combines both.

Metrics. jaccard_acc is the Jaccard index, also called example-based accuracy, which equals Equation 1 of Ibrohim and Budi (2019). subset_acc is subset accuracy, also called the exact match ratio. minority_macro_f1 is the macro F1 over the five minority labels. pred_hierarchy_violation_rate is the share of predicted label sets that break the annotation hierarchy.

Nested evaluation. Inner validation data are the 15 percent of each outer training fold that is held out for tuning. The outer fold is used only for testing.

Configuration IDs are built from these codes, for example svm__br__none__tuned. Transformer runs use the prefixes bert__mo, tf and ens. The IDs are kept unchanged so that stored runs are reused.

# Changes From the Previous Notebook

This notebook replaces Deteksi_Ujaran_Kebencian_MultiLabel.ipynb and reads and writes the same files in the results and figures folders.

All imports are in one place, and every component is a short function defined once, with its own heading. The previous notebook used classes and had parallel copies of the experiment runner, the grid runner, the threshold search, the seeding function, the transformer prediction function and the pairwise comparison code.

Every TF-IDF step now uses the deterministic TF-IDF. The previous notebook defined it but called the scikit-learn TfidfVectorizer in the main grid, the tuning step, the MLSMOTE inspection and the feature inspection. The stored results in experiments.csv were produced with the deterministic TF-IDF. Rerunning four configurations reproduces them exactly with it, and not with the scikit-learn one. The MLSMOTE inspection and the feature inspection now use the same features as the grid, which changes their numbers slightly.

The follow-up analyses use the algorithms that are present in the results. Before, running without a GPU dropped XGBoost and the MLP from those tables.

Manual notes in the cause column of results/error_analysis_false_negatives.csv are kept when the notebook is rerun. Before, the file was overwritten.

Pairwise comparisons also report the test statistic and an effect size, the matched-pairs rank-biserial correlation. These are extra columns at the end of each statistics table.

IRLbl and MeanIR are also reported for the deduplicated data in eda_label_counts_final.csv. The exploratory analysis describes the raw data before deduplication.

Derived tables use English column names, and their values are unchanged. All run switches are in the configuration cell.

# Configuration

All switches that decide which parts of the study run are in the cell below. A switch set to None is decided automatically from the available hardware.

In [ ]:
# Configuration. Edit values in this cell only.
RANDOM_STATE = 42
RESULTS_CSV = "experiments.csv"  # main grid, one row per configuration and fold
EXTENDED_CSV = "experiments_ext.csv"  # follow-up experiments
RUN_GRID = True  # classical grid, resumes automatically
RUN_GPU_MODELS = None  # XGBoost and MLP, None means run them when CUDA is available
RUN_BERT = None  # IndoBERTweet with the baseline protocol, None means automatic
RUN_EXTENDED_GRID = True  # ML-ROS and character n-gram grid, resumes automatically
RUN_TRANSFORMERS = None  # revised transformer protocol and ensembles, None means automatic
RUN_CROSS_CORPUS = None  # None means run when the IndoToxic2024 file is available
N_JOBS_CPU = None  # None means the number of CPUs minus one, at most four
BERT_MODEL = "indolem/indobertweet-base-uncased"
BERT_FALLBACK = "indobenchmark/indobert-base-p1"
DATA_REPO = (
    "https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection"
)

# Setup

This part imports the libraries, creates the folders, downloads the dataset, sets the random seeds and detects the computing device. PyTorch and transformers are optional. Without them the MLP and transformer parts are skipped and their stored results are used.

## Import Libraries

Every library used in the notebook is imported here. On Google Colab the packages that are missing there are installed first.

In [ ]:
import ast
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "iterative-stratification",
            "scikit-posthocs",
            "xgboost>=2.0",
            "emoji>=2.10",
        ],
        check=True,
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scikit_posthocs as skp
import scipy.sparse as sp
import seaborn as sns
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
from joblib import Parallel, delayed
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    jaccard_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

try:
    import torch
    from torch import nn
except ImportError:  # PyTorch is installed separately, see requirements.txt
    torch = nn = None
try:
    from transformers import (
        AutoModelForMaskedLM,
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
        get_linear_schedule_with_warmup,
    )
except ImportError:
    AutoModelForMaskedLM = AutoModelForSequenceClassification = AutoTokenizer = None
    DataCollatorForLanguageModeling = get_linear_schedule_with_warmup = None
try:
    import emoji
except ImportError:
    emoji = None

## Prepare Folders and Dataset

Folders for data, results and figures are created, and the dataset is cloned from the repository of the dataset authors on the first run.

In [ ]:
ROOT = Path.cwd()
RAW, PROC, EXTERNAL = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "data" / "external"
RES, FIG = ROOT / "results", ROOT / "figures"
for folder in [RAW, PROC, RES, FIG, RES / "preds", RES / "preds_ext"]:
    folder.mkdir(parents=True, exist_ok=True)

if not (RAW / "re_dataset.csv").exists():  # download the dataset on the first run
    source_dir = ROOT / "data" / "_src"
    if not source_dir.exists():
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1", DATA_REPO, str(source_dir)], check=True
        )
    for name in ["re_dataset.csv", "new_kamusalay.csv", "abusive.csv", "citation.bib"]:
        (RAW / name).write_bytes((source_dir / name).read_bytes())

## Set Seeds and Device

Random seeds and display options are set, and the computing device is detected. On a GPU, cuDNN runs in deterministic mode so that the MLP gives the same results on every run. Switches left at None in the configuration are decided from the device.

In [ ]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})

DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0)
    torch.backends.cudnn.deterministic = True  # reproducible MLP training
    torch.backends.cudnn.benchmark = False
else:
    GPU_NAME = "-" if torch is not None else "- (PyTorch not installed)"
if RUN_GPU_MODELS is None:
    RUN_GPU_MODELS = DEVICE == "cuda"
if RUN_BERT is None:
    RUN_BERT = DEVICE == "cuda"
if RUN_TRANSFORMERS is None:
    RUN_TRANSFORMERS = DEVICE == "cuda"
if N_JOBS_CPU is None:
    N_JOBS_CPU = max(1, min(4, (os.cpu_count() or 2) - 1))
print(
    f"ROOT={ROOT} | device={DEVICE} ({GPU_NAME}) | GPU models={RUN_GPU_MODELS} | "
    f"IndoBERTweet baseline={RUN_BERT} | revised transformers={RUN_TRANSFORMERS} | "
    f"CPU jobs={N_JOBS_CPU}"
)

# Labels and Annotation Hierarchy

The 12 labels follow a hierarchy that is checked against the data in the exploratory analysis. When HS is negative, every HS child label is negative. When HS is positive, a tweet has exactly one target, exactly one level and at least one category. HS_Other never occurs together with a specific category, and Abusive is independent of HS.

The rules are used in three places: to validate the data, to repair synthetic label sets inside MLSMOTE and to post-process predictions in the follow-up experiments.

## Label Groups

The 12 labels and their groups. The five minority labels are those whose imbalance ratio IRLbl is above the mean imbalance ratio MeanIR.

In [ ]:
LABELS = [
    "HS",
    "Abusive",
    "HS_Individual",
    "HS_Group",
    "HS_Religion",
    "HS_Race",
    "HS_Physical",
    "HS_Gender",
    "HS_Other",
    "HS_Weak",
    "HS_Moderate",
    "HS_Strong",
]
LABEL_INDEX = {name: i for i, name in enumerate(LABELS)}
TARGET_LABELS = ["HS_Individual", "HS_Group"]
SPECIFIC_CATEGORIES = ["HS_Religion", "HS_Race", "HS_Physical", "HS_Gender"]
OTHER_CATEGORY = "HS_Other"
LEVEL_LABELS = ["HS_Weak", "HS_Moderate", "HS_Strong"]
HS_CHILDREN = TARGET_LABELS + SPECIFIC_CATEGORIES + [OTHER_CATEGORY] + LEVEL_LABELS
CATEGORY_LABELS = SPECIFIC_CATEGORIES + [OTHER_CATEGORY]
MINORITY_LABELS = ["HS_Religion", "HS_Race", "HS_Physical", "HS_Gender", "HS_Strong"]

## Label Indices

Turns label names into column positions. The positions of each label group are computed once.

In [ ]:
def label_indices(names):
    return [LABEL_INDEX[name] for name in names]


HS_INDEX = LABEL_INDEX["HS"]
CHILD_INDICES = label_indices(HS_CHILDREN)
TARGET_INDICES = label_indices(TARGET_LABELS)
LEVEL_INDICES = label_indices(LEVEL_LABELS)
SPECIFIC_INDICES = label_indices(SPECIFIC_CATEGORIES)
OTHER_INDEX = LABEL_INDEX[OTHER_CATEGORY]
MINORITY_INDICES = label_indices(MINORITY_LABELS)

## Hierarchy Rule Masks

Returns one boolean mask per annotation rule. A row is true when it breaks that rule.

In [ ]:
def hierarchy_rule_masks(Y):
    Y = np.asarray(Y).astype(int)
    hs = Y[:, HS_INDEX]
    n_children = Y[:, CHILD_INDICES].sum(axis=1)
    n_targets = Y[:, TARGET_INDICES].sum(axis=1)
    n_levels = Y[:, LEVEL_INDICES].sum(axis=1)
    n_specific = Y[:, SPECIFIC_INDICES].sum(axis=1)
    other = Y[:, OTHER_INDEX]
    is_hs = hs == 1
    return {
        "hs0_but_child1": (hs == 0) & (n_children > 0),
        "hs1_target_not_exactly_one": is_hs & (n_targets != 1),
        "hs1_level_not_exactly_one": is_hs & (n_levels != 1),
        "hs1_no_category": is_hs & (n_specific + other == 0),
        "other_with_specific": (other == 1) & (n_specific > 0),
    }

## Hierarchy Violation Mask

Marks the rows that break at least one annotation rule.

In [ ]:
def hierarchy_violation_mask(Y):
    return np.logical_or.reduce(list(hierarchy_rule_masks(Y).values()))

## Count Hierarchy Violations

Counts the rows that break each rule and the rows that break any rule.

In [ ]:
def count_hierarchy_violations(Y):
    masks = hierarchy_rule_masks(Y)
    counts = {rule: int(mask.sum()) for rule, mask in masks.items()}
    counts["any_violation_rows"] = int(np.logical_or.reduce(list(masks.values())).sum())
    return counts

## Keep the Highest Label in a Group

Sets the label with the highest score in a group to positive and the rest of the group to negative. The row is changed in place.

In [ ]:
def keep_highest(y_row, scores, group):
    best = group[int(np.argmax(scores[group]))]
    y_row[group] = 0
    y_row[best] = 1

## Resolve Categories

Makes sure a hate speech row has at least one category. When HS_Other occurs together with a specific category, the side with the higher score is kept. The row is changed in place.

In [ ]:
def resolve_categories(y_row, scores):
    has_specific = y_row[SPECIFIC_INDICES].sum() > 0
    has_other = y_row[OTHER_INDEX] == 1
    if has_specific and has_other:
        if scores[OTHER_INDEX] > scores[SPECIFIC_INDICES].max():
            y_row[SPECIFIC_INDICES] = 0
        else:
            y_row[OTHER_INDEX] = 0
    elif not has_specific and not has_other:
        categories = SPECIFIC_INDICES + [OTHER_INDEX]
        y_row[categories[int(np.argmax(scores[categories]))]] = 1

## Repair Synthetic Label Set

Makes one synthetic label set from MLSMOTE follow the hierarchy. The scores are the label frequencies among the seed and its neighbours, and the label set of the seed breaks ties, so the repair follows the seed.

In [ ]:
def repair_synthetic_labelset(y, scores, seed_y):
    y = np.asarray(y).astype(int).copy()
    priority = np.asarray(scores, dtype=float) + 0.5 * np.asarray(seed_y).astype(
        int
    )  # the seed wins ties
    if y[CHILD_INDICES].sum() > 0:
        y[HS_INDEX] = 1
    if y[HS_INDEX] == 0:
        y[CHILD_INDICES] = 0
        return y
    keep_highest(y, priority, TARGET_INDICES)
    keep_highest(y, priority, LEVEL_INDICES)
    resolve_categories(y, priority)
    return y

## Enforce Label Hierarchy

Turns decision scores into label sets that follow the hierarchy. A positive score means the label is above its threshold. In topdown mode HS is decided by the HS score alone, and a negative HS clears every HS child label. In bottomup mode HS becomes positive when the HS score or any HS child score is positive. The rules come from the annotation guideline of the dataset, not from the test data.

In [ ]:
def enforce_label_hierarchy(S, mode="topdown"):
    S = np.asarray(S, dtype=float)
    Y = (S > 0).astype(int)
    hs = Y[:, HS_INDEX].astype(bool)
    if mode == "bottomup":
        hs = hs | (Y[:, CHILD_INDICES].sum(1) > 0)
    elif mode != "topdown":
        raise ValueError(mode)
    Y[:, HS_INDEX] = hs.astype(int)
    Y[np.ix_(~hs, CHILD_INDICES)] = 0
    for r in np.where(hs)[0]:
        keep_highest(Y[r], S[r], TARGET_INDICES)
        keep_highest(Y[r], S[r], LEVEL_INDICES)
        resolve_categories(Y[r], S[r])
    return Y

# Text Preprocessing

The classical models use the cleaned and slang normalised text. There is no stemming and no stopword removal, following Hana et al. (2020) and Hendrawan et al. (2020). The transformers receive the original tweet with the light normalisation used by IndoBERTweet (Koto et al., 2021).

## Load Raw Dataset

Loads re_dataset.csv. The file is latin-1 encoded, so UTF-8 decoding fails and the second encoding is used.

In [ ]:
def load_raw_dataset(path):
    try:
        df = pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
    missing = [c for c in ["Tweet"] + LABELS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    return df

## Load Slang Dictionary

Loads new_kamusalay.csv, which has no header and holds a slang word and its standard form on each row.

In [ ]:
def load_slang_dictionary(path):
    table = pd.read_csv(path, encoding="latin-1", header=None, names=["slang", "standard"])
    table = table.dropna().drop_duplicates("slang", keep="first")
    return dict(zip(table["slang"].str.strip(), table["standard"].str.strip()))

## Load Abusive Lexicon

Loads the list of abusive words distributed with the dataset. It is used only to mask abusive words in displayed examples.

In [ ]:
def load_abusive_lexicon(path):
    return set(pd.read_csv(path, encoding="latin-1")["ABUSIVE"].str.strip().str.lower())

## Remove Escaped Emoji and Newlines

The dataset stores emoji as escaped byte text and newlines as escaped text. Both are replaced by a space.

In [ ]:
_RE_ESCAPED_BYTES = re.compile(r"\\x[0-9a-fA-F]{2}")
_RE_ESCAPED_NEWLINE = re.compile(r"\\n")


def remove_escaped_bytes(text):
    return _RE_ESCAPED_NEWLINE.sub(" ", _RE_ESCAPED_BYTES.sub(" ", text))

## Remove URLs and HTML Entities

Web addresses and HTML entities such as the encoded ampersand are replaced by a space.

In [ ]:
_RE_HTTP = re.compile(r"https?://\S+|www\.\S+")
_RE_HTML_ENTITY = re.compile(r"&[a-z]+;")


def remove_urls_and_entities(text):
    return _RE_HTML_ENTITY.sub(" ", _RE_HTTP.sub(" ", text))

## Remove Twitter Tokens

The dataset replaces user names with USER and links with URL, and keeps RT for retweets. These tokens carry no meaning and are removed.

In [ ]:
_RE_SPECIAL_TOKENS = re.compile(r"\b(?:user|url|rt)\b")


def remove_twitter_tokens(text):
    return _RE_SPECIAL_TOKENS.sub(" ", text)

## Expand Reduplicated Words

Indonesian writes reduplication with the digit 2, so kita2 becomes kita kita and lucu2an becomes lucu lucuan.

In [ ]:
_RE_REDUPLICATION = re.compile(r"\b([a-z]{2,})2(?![0-9])")


def expand_reduplication(text):
    return _RE_REDUPLICATION.sub(r"\1 \1", text)

## Remove Digits and Punctuation

Digits are removed without splitting words, so g30spki becomes gspki. Punctuation, symbols and leftover non ASCII characters are then replaced by a space.

In [ ]:
_RE_DIGIT = re.compile(r"[0-9]+")
_RE_NON_ALPHA = re.compile(r"[^a-z\s]")


def remove_digits_and_punctuation(text):
    return _RE_NON_ALPHA.sub(" ", _RE_DIGIT.sub("", text))

## Collapse Repeated Letters

A letter repeated three or more times is reduced to one, so bangeeett becomes banget.

In [ ]:
_RE_REPEATED_CHAR = re.compile(r"([a-z])\1{2,}")


def collapse_repeated_letters(text):
    return _RE_REPEATED_CHAR.sub(r"\1", text)

## Clean Text

Runs the cleaning steps above in order after lowercasing, then removes extra spaces.

In [ ]:
_RE_SPACES = re.compile(r"\s+")


def clean_text(text):
    t = remove_escaped_bytes(str(text)).lower()
    t = remove_twitter_tokens(remove_urls_and_entities(t))
    t = remove_digits_and_punctuation(expand_reduplication(t))
    return _RE_SPACES.sub(" ", collapse_repeated_letters(t)).strip()

## Normalize Slang

Replaces each slang token with its standard form. A replacement can contain more than one word.

In [ ]:
def normalize_slang(text, slang_dict):
    return " ".join(slang_dict.get(token, token) for token in text.split())

## Build Preprocessor

Returns the preprocessing function for the classical models. Stemming and stopword removal exist only for ablation, are off by default and need the optional Sastrawi package.

In [ ]:
def build_preprocessor(slang_dict, use_stemming=False, use_stopwords=False):
    stemmer, stopwords = None, set()
    if use_stemming or use_stopwords:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

        stemmer = StemmerFactory().create_stemmer() if use_stemming else None
        stopwords = set(StopWordRemoverFactory().get_stop_words()) if use_stopwords else set()

    def preprocess(text):
        t = normalize_slang(clean_text(text), slang_dict)
        if stopwords:
            t = " ".join(w for w in t.split() if w not in stopwords)
        return stemmer.stem(t) if stemmer is not None else t

    return preprocess

## Normalize Text for Transformers

Light normalisation in the style of IndoBERTweet. Escaped bytes are decoded so that emoji become real characters, emoji are turned into words, USER becomes @USER and URL becomes HTTPURL.

In [ ]:
_RE_ESCAPED_BYTE_RUN = re.compile(r"(?:\\x[0-9a-fA-F]{2})+")


def decode_escaped_bytes(match):
    raw = bytes(int(h, 16) for h in re.findall(r"\\x([0-9a-fA-F]{2})", match.group(0)))
    return " " + raw.decode("utf-8", errors="ignore") + " "


def normalize_for_transformer(text):
    t = re.sub(r"^b['\"]", "", str(text)).rstrip("'\"")
    t = _RE_ESCAPED_BYTE_RUN.sub(decode_escaped_bytes, t.replace("\\n", " "))
    if emoji is not None:
        t = emoji.demojize(t, delimiters=(" ", " "))
    t = re.sub(r"\bRT\b", " ", t)
    t = re.sub(r"\bUSER\b", "@USER", t)
    t = re.sub(r"\bURL\b", "HTTPURL", t)
    return re.sub(r"\s+", " ", t).strip()

## Remove Duplicates

Duplicates of the cleaned text with identical labels keep one copy. Duplicates with different labels are annotation conflicts and are all dropped, because there is no basis for choosing the correct label set. Rows whose text is empty after cleaning are dropped. Duplicates spread over training and test folds would leak information, since the model could memorise the text.

In [ ]:
def deduplicate(df, text_col="text_clean"):
    n_raw = len(df)
    empty = df[text_col].str.len() == 0
    df = df[~empty]
    n_label_variants = df.groupby(text_col)[LABELS].nunique().max(axis=1)
    conflict_texts = set(n_label_variants[n_label_variants > 1].index)
    is_conflict = df[text_col].isin(conflict_texts)
    df_ok = df[~is_conflict]
    same_label_dup = df_ok.duplicated(text_col, keep="first")
    df_out = df_ok[~same_label_dup].reset_index(drop=True)
    report = {
        "rows_raw": n_raw,
        "rows_empty_after_clean": int(empty.sum()),
        "conflicting_texts": len(conflict_texts),
        "rows_dropped_conflict": int(is_conflict.sum()),
        "rows_dropped_same_label_dup": int(same_label_dup.sum()),
        "rows_final": len(df_out),
    }
    return df_out, report

## Mask Abusive Words

Masks abusive words for display, keeping only their first and last letter.

In [ ]:
def mask_abusive_words(text, abusive_lexicon):
    out = []
    for word in str(text).split():
        core = re.sub(r"[^a-zA-Z]", "", word).lower()
        if core in abusive_lexicon and len(core) > 2:
            out.append(word[0] + "*" * (len(word) - 2) + word[-1])
        else:
            out.append(word)
    return " ".join(out)

## Label Matrix

Returns the 12 labels of a data frame as an integer matrix.

In [ ]:
def label_matrix(df):
    return df[LABELS].to_numpy(dtype=int)

# Features

The scikit-learn TfidfVectorizer selects its vocabulary with an unstable sort. This dataset has hundreds of terms with equal frequency at the cut-off. In fold 0, 921 terms have frequency 6 and only 202 of them are kept, so results can differ between machines by up to 0.014 macro F1. The deterministic TF-IDF below uses the same criterion, the term frequency in the training texts, and breaks ties alphabetically, which makes the vocabulary reproducible.

## Deterministic TF-IDF

Keeps the most frequent training terms, breaks ties alphabetically and fits a TF-IDF with that fixed vocabulary.

In [ ]:
def fit_deterministic_tfidf(texts, ngram_range, max_features, sublinear_tf, analyzer, min_df=1):
    counter = CountVectorizer(ngram_range=ngram_range, min_df=min_df, analyzer=analyzer)
    counts = counter.fit_transform(texts)
    terms = counter.get_feature_names_out()
    term_freq = np.asarray(counts.sum(axis=0)).ravel()
    order = np.lexsort((terms, -term_freq))  # highest frequency first, ties in alphabetical order
    keep = order[:max_features] if max_features else order
    vocabulary = sorted(terms[keep].tolist())
    return TfidfVectorizer(
        ngram_range=ngram_range, sublinear_tf=sublinear_tf, analyzer=analyzer, vocabulary=vocabulary
    ).fit(texts)

## Feature Sets

Word features are unigrams and bigrams with 5,000 terms and sublinear term frequency. Character features are character 3 to 5 grams within word boundaries with 20,000 terms. The wordchar set stacks both. The function fits one TF-IDF per part of the chosen set on the training texts.

In [ ]:
WORD_TFIDF_PARAMS = {
    "ngram_range": (1, 2),
    "max_features": 5000,
    "sublinear_tf": True,
    "analyzer": "word",
}
CHAR_TFIDF_PARAMS = {
    "ngram_range": (3, 5),
    "max_features": 20000,
    "sublinear_tf": True,
    "analyzer": "char_wb",
}
FEATURE_SETS = {
    "word": [WORD_TFIDF_PARAMS],
    "char": [CHAR_TFIDF_PARAMS],
    "wordchar": [WORD_TFIDF_PARAMS, CHAR_TFIDF_PARAMS],
}


def fit_features(texts, features="word"):
    return [fit_deterministic_tfidf(texts, **params) for params in FEATURE_SETS[features]]

## Transform Features

Turns texts into a feature matrix with fitted TF-IDF models, stacking the parts side by side when there is more than one.

In [ ]:
def transform_features(vectorizers, texts):
    matrices = [vectorizer.transform(texts) for vectorizer in vectorizers]
    return matrices[0] if len(matrices) == 1 else sp.hstack(matrices, format="csr")

# Resampling

MLSMOTE (Charte et al., 2015, Knowledge-Based Systems) follows Algorithms 1 and 2 of the original paper. Minority labels are those whose IRLbl is above MeanIR, which is computed once. Neighbours are searched within the instances of each minority label, synthetic features interpolate the seed and a random neighbour, and synthetic label sets follow the ranking rule of the paper. Two additions are an optional hierarchy repair and a deterministic neighbour search.

ML-ROS (Charte et al., 2015, Neurocomputing) clones random instances of each minority label in turn until the clones reach 20 percent of the training set. This is close to the share that MLSMOTE adds, which is 20.6 percent on average.

## Imbalance Ratio per Label

IRLbl divides the count of the most frequent label by the count of each label. Labels without positive instances get infinity.

In [ ]:
def irlbl(Y):
    counts = np.asarray(Y).sum(axis=0).astype(float)
    with np.errstate(divide="ignore"):
        return np.where(counts > 0, counts.max() / counts, np.inf)

## Mean Imbalance Ratio

MeanIR is the mean of the finite IRLbl values.

In [ ]:
def mean_ir(Y):
    ratios = irlbl(Y)
    return float(ratios[np.isfinite(ratios)].mean())

## Nearest Neighbours

Finds the nearest neighbours of each instance by Euclidean distance. Distances are rounded to remove last digit floating point differences, and ties are broken by index, so the result is the same on every machine.

In [ ]:
def nearest_neighbours(X_bag, k):
    distances = np.round(euclidean_distances(X_bag), 9)
    np.fill_diagonal(distances, np.inf)
    return np.argsort(distances, axis=1, kind="stable")[:, :k]

## Synthetic Label Set

A label is given to a synthetic instance when it appears in more than half of the seed and its neighbours, which is the formula of Algorithm 2. The half threshold accepts exactly half, which is the wording in the text of the paper. With repair the label set is then made to follow the hierarchy.

In [ ]:
def synthetic_labelset(Y_bag, i, neighbours, threshold, repair):
    counts = Y_bag[i].astype(int) + Y_bag[neighbours].sum(axis=0)
    half = (len(neighbours) + 1) / 2.0
    y_new = ((counts > half) if threshold == "strict" else (counts >= half)).astype(np.int8)
    if not repair:
        return y_new, False
    repaired = np.asarray(repair_synthetic_labelset(y_new, counts, Y_bag[i])).astype(np.int8)
    return repaired, not np.array_equal(repaired, y_new)

## MLSMOTE for One Label

Creates one synthetic instance for every instance of one minority label. The features lie on the line between the seed and a random neighbour.

In [ ]:
def mlsmote_for_label(X_bag, Y_bag, k, threshold, repair, rng):
    neighbours = nearest_neighbours(X_bag, min(k, X_bag.shape[0] - 1))
    new_X, new_Y, n_repaired = [], [], 0
    for i in range(X_bag.shape[0]):
        neigh = list(neighbours[i])
        ref = neigh[rng.integers(len(neigh))]
        gap = rng.random()
        new_X.append(X_bag[i] + gap * (X_bag[ref] - X_bag[i]))
        y_new, was_repaired = synthetic_labelset(Y_bag, i, neigh, threshold, repair)
        new_Y.append(y_new)
        n_repaired += was_repaired
    return new_X, new_Y, n_repaired

## MLSMOTE

Visits the labels in order and oversamples each label that is still a minority label. IRLbl is updated after each label, while MeanIR stays at its starting value. Returns the resampled data and a summary.

In [ ]:
def mlsmote(X, Y, k=5, threshold="strict", repair=True, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    is_sparse = sp.issparse(X)
    stack = (lambda parts: sp.vstack(parts, format="csr")) if is_sparse else np.vstack
    X = sp.csr_matrix(X, dtype=np.float64) if is_sparse else np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(np.int8)
    m_ir = mean_ir(Y)  # computed once, as in Algorithm 1
    X_parts, Y_parts = [X], [Y]
    counts = Y.sum(axis=0).astype(float)
    summary = {
        "mean_ir": m_ir,
        "irlbl_before": irlbl(Y).tolist(),
        "processed_labels": [],
        "n_synthetic": 0,
        "n_repaired": 0,
        "n_skipped_small_bag": 0,
    }
    for label in range(Y.shape[1]):
        with np.errstate(divide="ignore"):
            ir_now = counts.max() / counts[label] if counts[label] > 0 else np.inf
        if not (ir_now > m_ir) or not np.isfinite(ir_now):
            continue
        Y_all = np.vstack(Y_parts)
        bag = np.where(Y_all[:, label] == 1)[0]
        if len(bag) < 2:
            summary["n_skipped_small_bag"] += 1
            continue
        new_X, new_Y, n_repaired = mlsmote_for_label(
            stack(X_parts)[bag], Y_all[bag], k, threshold, repair, rng
        )
        X_parts.append(stack(new_X))
        Y_parts.append(np.vstack(new_Y).astype(np.int8))
        counts += Y_parts[-1].sum(axis=0)
        summary["processed_labels"].append(int(label))
        summary["n_synthetic"] += len(new_Y)
        summary["n_repaired"] += n_repaired
    Y_out = np.vstack(Y_parts).astype(int)
    summary["irlbl_after"] = irlbl(Y_out).tolist()
    summary["mean_ir_after"] = mean_ir(Y_out)
    return stack(X_parts), Y_out, summary

## ML-ROS

Clones random instances of each minority label in turn. A label leaves the rotation once its IRLbl is no longer above MeanIR. Returns the resampled data and a summary.

In [ ]:
MLROS_PERCENTAGE = 0.20  # clones as a share of the training set


def mlros(X, Y, percentage=MLROS_PERCENTAGE, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    Y = np.asarray(Y).astype(int)
    m_ir = mean_ir(Y)
    counts = Y.sum(0).astype(float)
    ir_start = irlbl(Y)
    is_minority = np.isfinite(ir_start) & (ir_start > m_ir)
    bags = {j: np.where(Y[:, j] == 1)[0] for j in range(Y.shape[1]) if is_minority[j]}
    n_clones = int(round(percentage * len(Y)))
    picked = []
    while len(picked) < n_clones and bags:
        for j in list(bags):
            if len(picked) >= n_clones:
                break
            i = int(bags[j][rng.integers(len(bags[j]))])
            picked.append(i)
            counts += Y[i]
            if counts.max() / counts[j] <= m_ir:
                del bags[j]
    picked = np.asarray(picked, dtype=int)
    summary = {
        "n_synthetic": len(picked),
        "n_repaired": 0,
        "mean_ir": m_ir,
        "minority_labels": [LABELS[j] for j in np.where(is_minority)[0]],
    }
    if len(picked) == 0:
        return X, Y, summary
    X_out = sp.vstack([X, X[picked]], format="csr") if sp.issparse(X) else np.vstack([X, X[picked]])
    return X_out, np.vstack([Y, Y[picked]]), summary

## Resample Training Data

Applies the resampling of an imbalance treatment to training data. mlsmote uses the hierarchy repair, mlsmote_raw is the original algorithm and mlros is ML-ROS.

In [ ]:
RESAMPLING_CONDITIONS = {"mlsmote", "mlsmote_raw", "mlros"}


def resample(X, Y, condition):
    if condition == "mlros":
        return mlros(X, Y)
    return mlsmote(X, Y, k=5, threshold="strict", repair=(condition == "mlsmote"))

# Base Classifiers

The Bayesian model is Complement Naive Bayes, the support vector machine is LinearSVC, the tree ensemble is XGBoost and the neural network is an MLP written in PyTorch. XGBoost and the MLP use CUDA when it is available. The random forest is used only to replicate the Label Powerset baseline.

Class weighting means balanced sample weights for Complement Naive Bayes, balanced class weights for LinearSVC, scale_pos_weight for XGBoost and a positive class weight in the binary cross-entropy loss of the MLP.

## Hyperparameters and Display Names

Fixed hyperparameters, the small search spaces used for inner validation tuning, and the names shown in tables and figures.

In [ ]:
BASE_PARAMS = {
    "cnb": {"alpha": 1.0},
    "svm": {"C": 1.0, "max_iter": 5000, "random_state": RANDOM_STATE},
    "xgb": {
        "n_estimators": 300,
        "learning_rate": 0.1,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.5,
        "min_child_weight": 1,
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": 4,
        "verbosity": 0,
    },
    "mlp": {
        "hidden": 256,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-5,
        "batch_size": 128,
        "max_epochs": 30,
        "patience": 3,
        "val_fraction": 0.1,
        "random_state": RANDOM_STATE,
    },
    "rf": {
        "n_estimators": 100,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },  # Label Powerset baseline only
}
HYPERPARAMETER_GRID = {
    "cnb": [{"alpha": a} for a in (0.1, 0.3, 1.0)],
    "svm": [{"C": c} for c in (0.1, 0.3, 1.0, 3.0)],
    "xgb": [{}],  # thresholds only, to limit computation
    "mlp": [{}],  # thresholds only, to limit computation
    "rf": [{}],
}
ALGO_FAMILY = {
    "cnb": "Bayesian",
    "svm": "SVM",
    "xgb": "Tree ensemble",
    "mlp": "Neural network",
    "rf": "Tree ensemble",
    "bert": "Transformer",
    "tf": "Transformer",
    "ens": "Transformer",
}
ALGO_NAMES = {
    "cnb": "Complement NB",
    "svm": "LinearSVC",
    "xgb": "XGBoost",
    "mlp": "MLP",
    "rf": "Random Forest",
    "bert": "IndoBERTweet",
    "tf": "Transformer (revised protocol)",
    "ens": "Transformer ensemble",
}
IMBALANCE_NAMES = {
    "none": "No treatment",
    "weight": "Class weighting",
    "mlsmote": "MLSMOTE + repair",
    "mlsmote_raw": "MLSMOTE (original)",
    "mlros": "ML-ROS",
}

## MLP Batches

Yields the rows of a sparse matrix in batches as dense tensors on the device.

In [ ]:
def mlp_batches(X, idx, batch_size):
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start : start + batch_size]
        xb = X[batch_idx]
        xb = xb.toarray() if sp.issparse(xb) else np.asarray(xb)
        yield batch_idx, torch.from_numpy(xb.astype(np.float32)).to(DEVICE, non_blocking=True)

## MLP Validation Split

Holds out a stratified share of the training rows for early stopping.

In [ ]:
def mlp_validation_split(y, val_fraction, rng):
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n_val_pos = max(1, int(round(len(pos) * val_fraction)))
    n_val_neg = max(1, int(round(len(neg) * val_fraction)))
    val_idx = np.concatenate([rng.permutation(pos)[:n_val_pos], rng.permutation(neg)[:n_val_neg]])
    return np.setdiff1d(np.arange(len(y)), val_idx), val_idx

## MLP Training Epoch

One pass over the training rows in random order.

In [ ]:
def mlp_train_epoch(model, X, y_tensor, fit_idx, rng, batch_size, loss_fn, optimizer):
    model.train()
    for batch_idx, xb in mlp_batches(X, rng.permutation(fit_idx), batch_size):
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(xb).squeeze(1), y_tensor[batch_idx])
        loss.backward()
        optimizer.step()

## MLP Validation Loss

Mean loss on the held out rows.

In [ ]:
def mlp_validation_loss(model, X, y_tensor, val_idx, batch_size, loss_fn):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for batch_idx, xb in mlp_batches(X, val_idx, batch_size):
            total += loss_fn(model(xb).squeeze(1), y_tensor[batch_idx]).item() * len(batch_idx)
    return total / len(val_idx)

## Train MLP

Trains an MLP with one hidden layer for one binary label. Training stops when the validation loss has not improved for a number of epochs, and the best weights are kept.

In [ ]:
def train_mlp(
    X,
    y,
    weight=False,
    hidden=256,
    dropout=0.3,
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=128,
    max_epochs=30,
    patience=3,
    val_fraction=0.1,
    random_state=RANDOM_STATE,
):
    torch.manual_seed(random_state)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(random_state)
    rng = np.random.default_rng(random_state)
    X = sp.csr_matrix(X, dtype=np.float32)
    y = np.asarray(y).astype(np.float32)
    fit_idx, val_idx = mlp_validation_split(y, val_fraction, rng)
    model = nn.Sequential(
        nn.Linear(X.shape[1], hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
    ).to(DEVICE)
    pos_weight = None
    if weight:
        n_pos = max(y[fit_idx].sum(), 1.0)
        pos_weight = torch.tensor([(len(fit_idx) - n_pos) / n_pos], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y_tensor = torch.from_numpy(y).to(DEVICE)
    best_loss, best_state, bad_epochs = np.inf, None, 0
    for _ in range(max_epochs):
        mlp_train_epoch(model, X, y_tensor, fit_idx, rng, batch_size, loss_fn, optimizer)
        val_loss = mlp_validation_loss(model, X, y_tensor, val_idx, batch_size, loss_fn)
        if val_loss < best_loss - 1e-4:
            best_loss, bad_epochs = val_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## MLP Probabilities

Predicted probability of the positive class for every row.

In [ ]:
def mlp_probabilities(model, X, batch_size):
    X = sp.csr_matrix(X, dtype=np.float32)
    model.eval()
    out = []
    with torch.no_grad():
        for _, xb in mlp_batches(X, np.arange(X.shape[0]), batch_size):
            out.append(torch.sigmoid(model(xb).squeeze(1)).float().cpu().numpy())
    return np.concatenate(out)

## Make Estimator

Creates the scikit-learn or XGBoost model of an algorithm, with class weighting when it is requested.

In [ ]:
def make_estimator(algo, weight, y, params):
    params = dict(params)
    if algo == "cnb":
        return ComplementNB(**params)
    if algo == "svm":
        if weight:
            params["class_weight"] = "balanced"
        return LinearSVC(**params)
    if algo == "xgb":
        if weight:
            pos = max(int(y.sum()), 1)
            params["scale_pos_weight"] = (len(y) - pos) / pos
        return XGBClassifier(device=DEVICE, **params)
    if algo == "rf":
        return RandomForestClassifier(**params)
    raise ValueError(algo)

## Train Binary Model

Trains one binary model for one label. A label that is constant in the training data becomes a constant prediction, and the MLP is returned together with its batch size.

In [ ]:
def train_binary_model(algo, X, y, weight, override=None):
    if y.min() == y.max():
        return ("constant", int(y[0]))
    params = {**BASE_PARAMS[algo], **(override or {})}
    if algo == "mlp":
        return ("mlp", train_mlp(X, y, weight=weight, **params), params["batch_size"])
    estimator = make_estimator(algo, weight, y, params)
    if algo == "xgb":
        X = sp.csr_matrix(X, dtype=np.float32)
    if weight and algo == "cnb":
        estimator.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    else:
        estimator.fit(X, y)
    return estimator

## Binary Scores

Decision scores of one binary model, where a positive score means the label is predicted at the default threshold. LinearSVC uses its decision function and the other models use the probability minus 0.5.

In [ ]:
def binary_scores(model, X):
    if isinstance(model, tuple) and model[0] == "constant":
        return np.full(X.shape[0], 1.0 if model[1] == 1 else -1.0)
    if isinstance(model, tuple) and model[0] == "mlp":
        return mlp_probabilities(model[1], X, model[2]) - 0.5
    if isinstance(model, LinearSVC):
        return model.decision_function(X)
    if isinstance(model, XGBClassifier):
        X = sp.csr_matrix(X, dtype=np.float32)
    return model.predict_proba(X)[:, 1] - 0.5

# Multi-Label Strategies

Binary Relevance trains one independent binary model per label. Classifier Chains (Read et al., 2011) give each model the predictions for the earlier labels, and the chain follows the order of LABELS. Label Powerset treats every label combination as one class and is used only for the comparison with the baseline of Ibrohim and Budi (2019).

## Binary Relevance

One binary model per label, trained independently.

In [ ]:
def fit_binary_relevance(algo, X, Y, weight, override=None):
    return [train_binary_model(algo, X, Y[:, j], weight, override) for j in range(Y.shape[1])]

## Add Previous Labels

Appends the labels of the earlier links in the chain to the feature matrix.

In [ ]:
def with_previous_labels(X, Y_previous):
    if Y_previous.shape[1] == 0:
        return sp.csr_matrix(X)
    return sp.hstack([sp.csr_matrix(X), sp.csr_matrix(Y_previous.astype(np.float64))], format="csr")

## Classifier Chains

Each model in the chain is trained on the features plus the true values of the earlier labels.

In [ ]:
def fit_classifier_chain(algo, X, Y, weight, override=None):
    return [
        train_binary_model(algo, with_previous_labels(X, Y[:, :j]), Y[:, j], weight, override)
        for j in range(Y.shape[1])
    ]

## Chain Label Scores

Scores of one label in the chain, given the predictions for the earlier labels.

In [ ]:
def chain_label_scores(models, j, X, previous_predictions):
    return binary_scores(models[j], with_previous_labels(X, previous_predictions))

## Label Powerset

Every label combination seen in training is one class. Returns the trained model and the label set of each class.

In [ ]:
def fit_label_powerset(algo, X, Y, override=None):
    keys = ["".join(map(str, row)) for row in Y]
    classes, y = np.unique(keys, return_inverse=True)
    codes = np.array([[int(c) for c in key] for key in classes])
    params = {**BASE_PARAMS[algo], **(override or {})}
    estimator_class = {"cnb": ComplementNB, "svm": LinearSVC, "rf": RandomForestClassifier}[algo]
    return estimator_class(**params).fit(X, y), codes

## Fit Multi-Label Model

Trains a model with the chosen strategy and keeps the strategy name with it.

In [ ]:
def fit_multilabel(strategy, algo, X, Y, weight, override=None):
    if strategy == "br":
        return {"strategy": "br", "models": fit_binary_relevance(algo, X, Y, weight, override)}
    if strategy == "cc":
        return {"strategy": "cc", "models": fit_classifier_chain(algo, X, Y, weight, override)}
    if strategy == "lp":
        return {"strategy": "lp", "models": fit_label_powerset(algo, X, Y, override)}
    raise ValueError(strategy)

## Decision Scores

One score per label, where a positive score means the label is predicted. Thresholds are subtracted from the scores. In Classifier Chains each prediction feeds the next link, so the thresholds are applied along the chain.

In [ ]:
def decision_scores(model, X, thresholds=None):
    models = model["models"]
    if model["strategy"] == "lp":
        estimator, codes = models
        return np.where(codes[estimator.predict(X)] == 1, 1.0, -1.0)
    if model["strategy"] == "br":
        scores = np.column_stack([binary_scores(m, X) for m in models])
        return scores if thresholds is None else scores - np.asarray(thresholds)[None, :]
    t = np.zeros(len(models)) if thresholds is None else np.asarray(thresholds)
    scores = np.zeros((X.shape[0], len(models)))
    predictions = np.zeros((X.shape[0], len(models)), dtype=int)
    for j in range(len(models)):
        scores[:, j] = chain_label_scores(models, j, X, predictions[:, :j]) - t[j]
        predictions[:, j] = (scores[:, j] > 0).astype(int)
    return scores

# Inner Validation Tuning

Inside the training part of each outer fold, 15 percent of the data is held out as inner validation data with multi-label stratification. A small set of hyperparameters and one decision threshold per label are chosen there, and the model is then retrained on the whole outer training fold.

## Inner Validation Split

Splits training data into a fitting part and a 15 percent inner validation part with multi-label stratification.

In [ ]:
INNER_VALIDATION_SIZE = 0.15


def inner_validation_split(Y, seed=RANDOM_STATE):
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=INNER_VALIDATION_SIZE, random_state=seed
    )
    return next(splitter.split(np.zeros(len(Y)), Y))

## F1 Optimal Threshold

Searches 100 candidate thresholds, namely the 1st to 99th percentiles of the scores plus zero, and keeps the one with the highest F1. Among equally good candidates, the one closest to zero, the default threshold, is kept.

In [ ]:
def f1_optimal_threshold(y, s):
    y, s = np.asarray(y), np.asarray(s)
    if y.sum() == 0:
        return 0.0
    candidates = np.unique(np.concatenate([np.quantile(s, np.linspace(0.01, 0.99, 99)), [0.0]]))
    predicted = s[None, :] > candidates[:, None]
    tp = (predicted & (y[None, :] == 1)).sum(1)
    fp = (predicted & (y[None, :] == 0)).sum(1)
    fn = y.sum() - tp
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0)
    best = candidates[np.abs(f1 - f1.max()) <= 1e-12]
    return float(best[np.argmin(np.abs(best))])

## Tune Label Thresholds

One threshold per label on the decision scores of a trained model. For Classifier Chains the thresholds are chosen in chain order, because each prediction feeds the next link.

In [ ]:
def tune_label_thresholds(model, X_val, Y_val):
    n_labels = Y_val.shape[1]
    if model["strategy"] != "cc":
        S = decision_scores(model, X_val)
        return np.array([f1_optimal_threshold(Y_val[:, j], S[:, j]) for j in range(n_labels)])
    thresholds = np.zeros(n_labels)
    predictions = np.zeros_like(Y_val)
    for j in range(n_labels):
        s = chain_label_scores(model["models"], j, X_val, predictions[:, :j])
        thresholds[j] = f1_optimal_threshold(Y_val[:, j], s)
        predictions[:, j] = (s > thresholds[j]).astype(int)
    return thresholds

## Tune Probability Thresholds

The same threshold search on the probability scale, used for the transformers.

In [ ]:
def tune_probability_thresholds(Y, P):
    return np.array([f1_optimal_threshold(Y[:, j], P[:, j] - 0.5) for j in range(Y.shape[1])]) + 0.5

## Select on Inner Validation

Tries every hyperparameter candidate on the inner split, tunes the thresholds for each and keeps the candidate with the highest inner validation macro F1. Returns the chosen hyperparameters, the thresholds and a log of the search.

In [ ]:
def select_on_inner_validation(algo, strategy, condition, texts_train, Y_train, features="word"):
    i_fit, i_val = inner_validation_split(Y_train)
    vectorizers = fit_features(texts_train.iloc[i_fit], features)
    X_fit = transform_features(vectorizers, texts_train.iloc[i_fit])
    X_val = transform_features(vectorizers, texts_train.iloc[i_val])
    Y_fit, Y_val = Y_train[i_fit], Y_train[i_val]
    if condition in RESAMPLING_CONDITIONS:
        X_fit, Y_fit, _ = resample(X_fit, Y_fit, condition)
    log, best = [], None
    for override in HYPERPARAMETER_GRID[algo]:
        model = fit_multilabel(strategy, algo, X_fit, Y_fit, condition == "weight", override)
        thresholds = (
            np.zeros(Y_train.shape[1])
            if strategy == "lp"
            else tune_label_thresholds(model, X_val, Y_val)
        )
        predictions = (decision_scores(model, X_val, thresholds) > 0).astype(int)
        score = f1_score(Y_val, predictions, average="macro", zero_division=0)
        log.append({"override": override, "inner_macro_f1": round(float(score), 4)})
        if best is None or score > best[0]:
            best = (score, override, thresholds)
    return best[1], best[2], log

# Evaluation Metrics

The label-based metrics are macro F1, micro F1, the macro F1 over the five minority labels, and precision, recall and F1 for each label. The example-based metrics are Hamming loss, the Jaccard index and subset accuracy. The Jaccard index per document equals Equation 1 of Ibrohim and Budi (2019), so it can be compared with their baseline of 66.12 percent. A document with no true labels and no predicted labels counts as correct. The hierarchy violation rate is the share of predicted label sets that break the annotation rules.

## Evaluate Predictions

Computes every metric for one set of predictions.

In [ ]:
def evaluate_predictions(Y_true, Y_pred):
    Y_true = np.asarray(Y_true).astype(int)
    Y_pred = np.asarray(Y_pred).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        Y_true, Y_pred, average=None, zero_division=0
    )
    out = {
        "macro_f1": f1_score(Y_true, Y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(Y_true, Y_pred, average="micro", zero_division=0),
        "minority_macro_f1": float(np.mean([f1[i] for i in MINORITY_INDICES])),
        "hamming_loss": hamming_loss(Y_true, Y_pred),
        "jaccard_acc": jaccard_score(Y_true, Y_pred, average="samples", zero_division=1),
        "subset_acc": accuracy_score(Y_true, Y_pred),
        "pred_hierarchy_violation_rate": float(hierarchy_violation_mask(Y_pred).mean()),
    }
    for j, label in enumerate(LABELS):
        out[f"f1__{label}"] = f1[j]
        out[f"p__{label}"] = precision[j]
        out[f"r__{label}"] = recall[j]
    return out

## Macro F1 at Thresholds

A fast macro F1 for probabilities and thresholds, counting a label without true or predicted positives as zero.

In [ ]:
def macro_f1_at(Y, P, thresholds):
    predicted = P > thresholds
    tp = (predicted & (Y == 1)).sum(0)
    fp = (predicted & (Y == 0)).sum(0)
    fn = (~predicted & (Y == 1)).sum(0)
    return float(
        np.mean(np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0))
    )

# Statistical Tests

Paired scores are compared with the Wilcoxon signed-rank test, and the matched-pairs rank-biserial correlation is reported as its effect size. Per-fold macro F1 is compared with the corrected resampled t-test of Nadeau and Bengio (2003), which accounts for the overlap between cross-validation training sets. The Holm-Bonferroni correction is applied within each family of comparisons. Algorithms are compared with the Friedman test, the Nemenyi post-hoc test and critical difference diagrams (Demšar, 2006).

## Corrected Resampled t-Test

Paired test on per-fold scores whose variance is corrected for the overlap between training sets.

In [ ]:
def corrected_resampled_ttest(a, b, n_train, n_test):
    d = np.asarray(a, float) - np.asarray(b, float)
    k = len(d)
    var = d.var(ddof=1)
    if var == 0:
        return {"t": 0.0, "p": 1.0}
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * var)
    return {"t": float(t), "p": float(2 * stats.t.sf(abs(t), df=k - 1))}

## Wilcoxon Signed-Rank Test

Paired test on scores. The rank-biserial correlation is positive when the first set of scores tends to be larger.

In [ ]:
def wilcoxon_signed_rank(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if np.allclose(a, b):
        return {"stat": 0.0, "p": 1.0, "rank_biserial": 0.0}
    result = stats.wilcoxon(a, b, zero_method="wilcox")
    d = (a - b)[a != b]
    ranks = stats.rankdata(np.abs(d))
    rank_biserial = (ranks[d > 0].sum() - ranks[d < 0].sum()) / ranks.sum()
    return {
        "stat": float(result.statistic),
        "p": float(result.pvalue),
        "rank_biserial": float(rank_biserial),
    }

## Holm-Bonferroni Correction

Adjusts a list of p-values for multiple comparisons.

In [ ]:
def holm_bonferroni(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adjusted = np.empty(len(p))
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adjusted[i] = min(running, 1.0)
    return adjusted

## Add Holm Columns

Adds a corrected copy of each p-value column, corrected within each group of a column or over all rows.

In [ ]:
def add_holm_columns(df, p_columns, by=None):
    groups = (
        [np.ones(len(df), dtype=bool)]
        if by is None
        else [(df[by] == g).to_numpy() for g in df[by].unique()]
    )
    for mask in groups:
        for col in p_columns:
            df.loc[mask, f"{col}_holm"] = holm_bonferroni(df.loc[mask, col].to_numpy())
    return df

## Friedman and Nemenyi Tests

The rows of the score table are blocks, here the labels, and the columns are methods. Higher scores are better.

In [ ]:
def friedman_nemenyi(M):
    chi2, p = stats.friedmanchisquare(*[M[c].to_numpy() for c in M.columns])
    nemenyi = skp.posthoc_nemenyi_friedman(M.to_numpy())
    nemenyi.index = nemenyi.columns = M.columns
    avg_rank = M.rank(axis=1, ascending=False).mean().to_dict()
    return {"chi2": float(chi2), "p": float(p), "nemenyi": nemenyi, "avg_rank": avg_rank}

## Paired Comparison

Compares configuration B with configuration A. The Wilcoxon test uses the 12 per-label F1 pairs and the corrected resampled t-test uses the per-fold macro F1 pairs.

In [ ]:
EFFECT_SIZE_COLUMNS = ["wilcoxon_stat", "rank_biserial", "corrected_t"]


def paired_comparison(f1_a, f1_b, folds_a, folds_b, n_train, n_test):
    f1_a, f1_b = np.asarray(f1_a), np.asarray(f1_b)
    wilcoxon = wilcoxon_signed_rank(f1_b, f1_a)
    ttest = corrected_resampled_ttest(folds_b, folds_a, n_train, n_test)
    return {
        "macro_f1_a": f1_a.mean(),
        "macro_f1_b": f1_b.mean(),
        "delta_macro_f1": f1_b.mean() - f1_a.mean(),
        "delta_minority_f1": f1_b[MINORITY_INDICES].mean() - f1_a[MINORITY_INDICES].mean(),
        "n_labels_improved": int((f1_b > f1_a).sum()),
        "wilcoxon_p": wilcoxon["p"],
        "corrected_t_p": ttest["p"],
        "wilcoxon_stat": wilcoxon["stat"],
        "rank_biserial": wilcoxon["rank_biserial"],
        "corrected_t": ttest["t"],
    }

## Move Columns Last

Moves the given columns to the end of a table, used to keep the effect sizes after the p-values.

In [ ]:
def move_columns_last(df, columns):
    present = [c for c in columns if c in df.columns]
    return df[[c for c in df.columns if c not in present] + present]

# Experiment Runner

One run is one combination of algorithm, strategy, imbalance treatment, tuning, features and fold. Rows are appended to results/experiments.csv for the main grid or to results/experiments_ext.csv for the follow-up experiments. The decision scores on the test fold are stored in results/preds or results/preds_ext. Runs that are already in the results file are skipped. CPU models run in parallel and GPU models run one at a time.

## Results File Layout

The column order of the two run logs, used when a log file is created from scratch.

In [ ]:
GPU_ALGOS = {"xgb", "mlp"}
METRIC_COLUMNS = [
    "macro_f1",
    "micro_f1",
    "minority_macro_f1",
    "hamming_loss",
    "jaccard_acc",
    "subset_acc",
    "pred_hierarchy_violation_rate",
] + [f"{metric}__{label}" for label in LABELS for metric in ("f1", "p", "r")]
RESULT_COLUMNS = {
    RESULTS_CSV: [
        "config",
        "algo",
        "strategy",
        "condition",
        "tuned",
        "fold",
        "n_train",
        "n_test",
        "device",
        "train_time_s",
        "tune_time_s",
        "total_time_s",
        "n_synthetic",
        "n_repaired",
        "override",
        "thresholds",
        "inner_log",
    ]
    + METRIC_COLUMNS,
    EXTENDED_CSV: [
        "config",
        "algo",
        "strategy",
        "condition",
        "features",
        "tuned",
        "fold",
        "n_train",
        "n_test",
        "device",
        "train_time_s",
        "tune_time_s",
        "total_time_s",
        "n_synthetic",
        "override",
        "thresholds",
        "inner_log",
        "inner_macro_f1",
        "best_epoch",
    ]
    + METRIC_COLUMNS,
}

## Configuration ID

Builds the identifier of a run, for example svm__br__none__tuned. The features are added only when they are not word.

In [ ]:
def config_id(algo, strategy, condition, tuned=False, features="word"):
    feature_part = "" if features == "word" else f"__{features}"
    return f"{algo}__{strategy}__{condition}{feature_part}" + ("__tuned" if tuned else "")

## Load Inputs

Reads the deduplicated data and the stored folds.

In [ ]:
def load_inputs(root):
    data = pd.read_csv(root / "data" / "processed" / "data_clean.csv", keep_default_na=False)
    folds = json.loads((root / "results" / "folds.json").read_text())
    return data, folds

## Fold Rows

Turns the training and test ids of one fold into row positions.

In [ ]:
def fold_rows(data, fold):
    row_of_id = pd.Series(np.arange(len(data)), index=data["id"])
    return row_of_id[fold["train_ids"]].to_numpy(), row_of_id[fold["test_ids"]].to_numpy()

## Completed Runs

The configuration and fold of every run already recorded in a run log.

In [ ]:
def completed_runs(path):
    if not path.exists():
        return set()
    previous = pd.read_csv(path)
    return set(zip(previous.config, previous.fold))

## Append Rows

Appends result rows to a run log, keeping the column order of the existing file.

In [ ]:
def append_rows(path, rows):
    df = pd.DataFrame(rows)
    if path.exists():
        columns = pd.read_csv(path, nrows=0).columns.tolist()
    else:
        columns = RESULT_COLUMNS.get(path.name, df.columns.tolist())
    df.reindex(columns=columns).to_csv(path, mode="a", header=not path.exists(), index=False)

## Train and Score

Fits the features on the training fold, resamples the training fold when the imbalance treatment asks for it, trains the model and scores the test fold. Nothing is fitted on the test fold.

In [ ]:
def train_and_score(
    algo, strategy, condition, override, thresholds, texts_train, Y_train, texts_test, features
):
    vectorizers = fit_features(texts_train, features)
    X_train = transform_features(vectorizers, texts_train)
    X_test = transform_features(vectorizers, texts_test)
    resample_summary = {}
    if condition in RESAMPLING_CONDITIONS:
        X_train, Y_train, resample_summary = resample(X_train, Y_train, condition)
    t0 = time.time()
    model = fit_multilabel(strategy, algo, X_train, Y_train, condition == "weight", override)
    train_time = time.time() - t0
    return (
        decision_scores(model, X_test, thresholds),
        X_train.shape[0],
        train_time,
        resample_summary,
    )

## Run One Experiment

Runs one configuration on one fold, with inner validation tuning when it is requested, and returns the result row. The test scores are saved for later analysis.

In [ ]:
def run_experiment(
    root, algo, strategy, condition, tuned, features, fold_id, data, folds, preds_subdir="preds"
):
    train_rows, test_rows = fold_rows(data, folds[fold_id])
    Y, texts = label_matrix(data), data["text_clean"]
    t0 = time.time()
    override, thresholds, tuning_log = None, None, []
    if tuned:
        override, thresholds, tuning_log = select_on_inner_validation(
            algo,
            strategy,
            condition,
            texts.iloc[train_rows].reset_index(drop=True),
            Y[train_rows],
            features,
        )
    tune_time = time.time() - t0
    scores, n_train, train_time, resample_summary = train_and_score(
        algo,
        strategy,
        condition,
        override,
        thresholds,
        texts.iloc[train_rows],
        Y[train_rows],
        texts.iloc[test_rows],
        features,
    )
    row = {
        "config": config_id(algo, strategy, condition, tuned, features),
        "algo": algo,
        "strategy": strategy,
        "condition": condition,
        "features": features,
        "tuned": bool(tuned),
        "fold": fold_id,
        "n_train": n_train,
        "n_test": len(test_rows),
        "device": DEVICE if algo in GPU_ALGOS else "cpu",
        "train_time_s": round(train_time, 2),
        "tune_time_s": round(tune_time, 2),
        "total_time_s": round(time.time() - t0, 2),
        "n_synthetic": resample_summary.get("n_synthetic", 0),
        "n_repaired": resample_summary.get("n_repaired", 0),
        "override": json.dumps(override or {}),
        "thresholds": (
            json.dumps([round(float(x), 4) for x in thresholds]) if thresholds is not None else ""
        ),
        "inner_log": json.dumps(tuning_log),
        "inner_macro_f1": max([r["inner_macro_f1"] for r in tuning_log], default=np.nan),
    }
    row.update(evaluate_predictions(Y[test_rows], (scores > 0).astype(int)))
    np.savez_compressed(
        root / "results" / preds_subdir / f"{row['config']}__f{fold_id}.npz",
        test_ids=np.asarray(folds[fold_id]["test_ids"]),
        scores=scores.astype(np.float32),
    )
    return row

## Build Grid

Lists every combination of algorithm, strategy, imbalance treatment, tuning, features and fold.

In [ ]:
def build_grid(algos, strategies, conditions, tuned=(False,), features=("word",), n_folds=5):
    return [
        (a, s, c, t, ft, k)
        for a in algos
        for s in strategies
        for c in conditions
        for t in tuned
        for ft in features
        for k in range(n_folds)
    ]

## Print Run Summary

Prints one line per finished run.

In [ ]:
def print_run(row):
    print(
        f"{row['config']:<36} f{row['fold']} macro-F1={row['macro_f1']:.3f} "
        f"minority={row['minority_macro_f1']:.3f} jaccard={row['jaccard_acc']:.3f} "
        f"{row['total_time_s']:.0f}s [{row['device']}]",
        flush=True,
    )

## Run Grid

Runs every entry of the grid that is not yet in the results file. CPU models run in parallel batches and GPU models run one at a time.

In [ ]:
def run_grid(root, grid, n_jobs_cpu=2, csv_name=RESULTS_CSV, preds_subdir="preds", verbose=True):
    out = Path(root) / "results" / csv_name
    (Path(root) / "results" / preds_subdir).mkdir(parents=True, exist_ok=True)
    done = completed_runs(out)
    todo = [g for g in grid if (config_id(*g[:5]), g[5]) not in done]
    print(
        f"{csv_name}: {len(grid)} runs in grid, {len(grid) - len(todo)} done, {len(todo)} to run "
        f"| device: {DEVICE}",
        flush=True,
    )
    if not todo:
        return pd.read_csv(out)
    data, folds = load_inputs(Path(root))

    def job(g):
        return run_experiment(Path(root), *g, data=data, folds=folds, preds_subdir=preds_subdir)

    def record(rows):
        append_rows(out, rows)
        if verbose:
            for row in rows:
                print_run(row)

    cpu_runs = [g for g in todo if g[0] not in GPU_ALGOS]
    batch = max(n_jobs_cpu, 1) * 2
    for i in range(0, len(cpu_runs), batch):
        record(Parallel(n_jobs=n_jobs_cpu)(delayed(job)(g) for g in cpu_runs[i : i + batch]))
    for g in [g for g in todo if g[0] in GPU_ALGOS]:
        record([job(g)])
    return pd.read_csv(out)

## Load Complete Runs

Reads a run log, drops configurations without all five folds and adds a tuning column with fixed or tuned.

In [ ]:
def load_complete_runs(path):
    df = pd.read_csv(path)
    n_folds = df.groupby("config").fold.nunique()
    incomplete = n_folds[n_folds < 5]
    if len(incomplete):
        print(
            f"WARNING: configurations without five folds in {path.name} (ignored):",
            incomplete.to_dict(),
        )
        df = df[~df.config.isin(incomplete.index)].copy()
    df["tuning"] = np.where(df.tuned.astype(bool), "tuned", "fixed")
    return df

## Load Fold Scores

Returns the test ids and decision scores of one fold, where a positive score is a positive prediction. Transformer runs store probabilities, so their thresholds are subtracted here.

In [ ]:
def load_fold_scores(config, k):
    path = RES / "preds" / f"{config}__f{k}.npz"
    if path.exists():
        z = np.load(path)
        return z["test_ids"], z["scores"]
    base = config[: -len("__tuned")] if config.endswith("__tuned") else config
    if base.startswith(("tf__", "ens__")):
        z = np.load(RES / "preds_ext" / f"{base}__f{k}.npz")
        thresholds = z["thresholds"] if config.endswith("__tuned") else 0.5
        return (
            z["test_ids"],
            z["test_prob"] - np.atleast_1d(np.asarray(thresholds, dtype=float))[None, :],
        )
    z = np.load(RES / "preds_ext" / f"{config}__f{k}.npz")
    return z["test_ids"], z["scores"]

## Save Table

Rounds a derived table, renames the condition column to imbalance_treatment and writes the table to the results folder.

In [ ]:
COLUMN_RENAMES = {"condition": "imbalance_treatment"}


def save_table(df, name, index=True, decimals=4):
    out = df.round(decimals) if decimals is not None else df.copy()
    out = out.rename(columns=COLUMN_RENAMES)
    if index:
        out.index = out.index.set_names([COLUMN_RENAMES.get(n, n) for n in out.index.names])
    out.to_csv(RES / name, index=index)
    return out

# Transformer Fine-Tuning

Two fine-tuning protocols share the functions in this part. Both hold out 15 percent of each outer training fold as inner validation data.

The baseline protocol trains IndoBERTweet on the cleaned text for at most four epochs, with early stopping on inner validation macro F1 at threshold 0.5. Class weighting uses the ratio of negative to positive examples.

The revised protocol uses the original tweets with IndoBERTweet normalisation. It selects the epoch by inner validation macro F1 after threshold tuning, with at most 6 or 10 epochs and a patience of 2. Its positive class weight is the square root of the negative to positive ratio, capped at 10, and it stores the inner validation probabilities for ensembles and threshold selection.

## Set Global Seed

Seeds Python, NumPy and PyTorch.

In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Encode Texts

Tokenises all texts once, padded to the same length.

In [ ]:
def encode_texts(tokenizer, texts, max_len):
    enc = tokenizer(
        list(texts), truncation=True, max_length=max_len, padding="max_length", return_tensors="pt"
    )
    return {k: v for k, v in enc.items() if k in ("input_ids", "attention_mask", "token_type_ids")}

## Predict Probabilities

Sigmoid outputs for the given rows of the encoded texts, computed in batches with mixed precision on the GPU.

In [ ]:
def predict_probabilities(model, enc, idx, batch_size, device):
    model.eval()
    out = []
    with torch.inference_mode():
        for start in range(0, len(idx), batch_size):
            batch_idx = torch.as_tensor(idx[start : start + batch_size])
            batch = {k: v[batch_idx].to(device) for k, v in enc.items()}
            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")
            ):
                logits = model(**batch).logits
            out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

## Training Tools

Creates the AdamW optimizer, the linear schedule with warmup and the gradient scaler for mixed precision.

In [ ]:
def make_training_tools(model, n_rows, batch_size, epochs, lr, weight_decay, warmup, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps = math.ceil(n_rows / batch_size) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(warmup * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    return optimizer, scheduler, scaler

## Train One Epoch

One pass over the given rows in random order, with mixed precision and gradient clipping. Returns the summed loss.

In [ ]:
def train_one_epoch(model, enc, Y_tensor, rows, rng, batch_size, loss_fn, tools, device):
    optimizer, scheduler, scaler = tools
    model.train()
    order = rng.permutation(rows)
    total = 0.0
    for start in range(0, len(order), batch_size):
        batch_idx = torch.as_tensor(order[start : start + batch_size])
        batch = {k: v[batch_idx].to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), Y_tensor[batch_idx].to(device))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += float(loss.detach())
    return total

## Copy Model State

A copy of the model weights on the CPU, used to keep the best epoch.

In [ ]:
def copy_model_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

## Positive Class Weights

Weights for the positive class in the binary cross-entropy loss. ratio gives the ratio of negative to positive examples, sqrt10 gives its square root capped at 10 and none gives no weights.

In [ ]:
def positive_class_weights(Y, mode, device):
    if mode == "none":
        return None
    pos = Y.sum(0).clip(min=1)
    ratio = (len(Y) - pos) / pos
    if mode == "sqrt10":
        ratio = np.minimum(np.sqrt(ratio), 10.0)
    return torch.tensor(ratio, dtype=torch.float32, device=device)

## Asymmetric Loss

The asymmetric loss for multi-label classification of Ridnik et al. (2021). It is available as an option of the revised protocol.

In [ ]:
def asymmetric_loss(logits, y, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
    p = torch.sigmoid(logits)
    p_neg = (p - clip).clamp(min=0) if clip else p
    loss_pos = y * torch.log(p.clamp(min=eps)) * (1 - p) ** gamma_pos
    loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=eps)) * p_neg**gamma_neg
    return -(loss_pos + loss_neg).sum(1).mean()

## Baseline Protocol Settings

Settings of the baseline protocol. A batch size of 0 means it is chosen from the GPU memory.

In [ ]:
BERT_BASELINE_CONFIG = {
    "model": BERT_MODEL,
    "fallback_model": BERT_FALLBACK,
    "max_len": 128,
    "epochs": 4,
    "lr": 2e-5,
    "batch": 0,
    "patience": 1,
}

## Load Classifier With Fallback

Loads the first model that can be loaded from a list of model names.

In [ ]:
def load_classifier_with_fallback(names, n_labels):
    for name in names:
        try:
            tokenizer = AutoTokenizer.from_pretrained(name)
            model = AutoModelForSequenceClassification.from_pretrained(
                name, num_labels=n_labels, problem_type="multi_label_classification"
            )
            return model, tokenizer, name
        except Exception as e:
            print(f"Could not load {name}: {e!r}", flush=True)
    raise RuntimeError("No model could be loaded")

## Auto Batch Size

Chooses the batch size from the GPU memory.

In [ ]:
def auto_batch_size(device):
    if device != "cuda":
        return 16
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return 32 if vram_gb >= 10 else 16 if vram_gb >= 6 else 8

## Train Baseline Epochs

Trains for at most the configured number of epochs and stops when the inner validation macro F1 at threshold 0.5 stops improving. The weights of the best epoch are loaded at the end.

In [ ]:
def train_baseline_epochs(cfg, model, enc, Y, fit_rows, val_rows, batch_size, loss_fn, device, tag):
    Y_tensor = torch.from_numpy(Y)
    tools = make_training_tools(
        model, len(fit_rows), batch_size, cfg["epochs"], cfg["lr"], 0.01, 0.1, device
    )
    t0 = time.time()
    best_f1, best_state, bad_epochs = -1, None, 0
    rng = np.random.default_rng(RANDOM_STATE)
    for epoch in range(cfg["epochs"]):
        train_one_epoch(model, enc, Y_tensor, fit_rows, rng, batch_size, loss_fn, tools, device)
        p_val = predict_probabilities(model, enc, val_rows, batch_size * 2, device)
        f1_val = f1_score(
            Y[val_rows].astype(int), (p_val > 0.5).astype(int), average="macro", zero_division=0
        )
        print(
            f"  {tag} epoch {epoch + 1}: inner macro-F1={f1_val:.4f} "
            f"({(time.time() - t0) / 60:.1f} min)",
            flush=True,
        )
        if f1_val > best_f1 + 1e-4:
            best_f1, bad_epochs, best_state = f1_val, 0, copy_model_state(model)
        else:
            bad_epochs += 1
            if bad_epochs > cfg["patience"] - 1:
                break
    model.load_state_dict(best_state)
    return best_f1, epoch + 1, time.time() - t0

## Baseline Result Rows

Builds one row with default thresholds and one row with tuned thresholds, and saves the test scores of both.

In [ ]:
def baseline_result_rows(
    root, data, condition, fold_id, test_rows, scores_by_tuning, thresholds, info
):
    rows = []
    for tuned, scores in scores_by_tuning:
        cfg_id = f"bert__mo__{condition}" + ("__tuned" if tuned else "")
        row = {
            "config": cfg_id,
            "algo": "bert",
            "strategy": "mo",
            "condition": condition,
            "tuned": tuned,
            "fold": fold_id,
            "n_train": info["n_train"],
            "n_test": len(test_rows),
            "device": info["device"],
            "train_time_s": round(info["train_time"], 1),
            "tune_time_s": 0.0,
            "total_time_s": round(time.time() - info["t0"], 1),
            "n_synthetic": 0,
            "n_repaired": 0,
            "override": json.dumps(info["override"]),
            "thresholds": json.dumps([round(float(x), 4) for x in thresholds]) if tuned else "",
            "inner_log": json.dumps({"best_val_macro_f1": round(info["best_f1"], 4)}),
        }
        row.update(evaluate_predictions(info["Y_test"], (scores > 0).astype(int)))
        rows.append(row)
        np.savez_compressed(
            root / "results" / "preds" / f"{cfg_id}__f{fold_id}.npz",
            test_ids=np.asarray(data["id"].to_numpy()[test_rows]),
            scores=scores.astype(np.float32),
        )
    return rows

## Run Baseline Fold

Runs the baseline protocol on one outer fold and appends the two result rows.

In [ ]:
def run_bert_baseline_fold(
    root, cfg, data, folds, fold_id, condition, device, done, csv_name=RESULTS_CSV
):
    config_plain = f"bert__mo__{condition}"
    if (config_plain, fold_id) in done and (config_plain + "__tuned", fold_id) in done:
        print(f"skip {config_plain} f{fold_id} (already done)", flush=True)
        return
    set_global_seed(RANDOM_STATE)
    train_rows, test_rows = fold_rows(data, folds[fold_id])
    Y = data[LABELS].to_numpy(dtype=np.float32)
    i_fit, i_val = inner_validation_split(Y[train_rows].astype(int))
    fit_rows, val_rows = train_rows[i_fit], train_rows[i_val]
    model, tokenizer, model_name = load_classifier_with_fallback(
        (cfg["model"], cfg["fallback_model"]), len(LABELS)
    )
    model.to(device)
    enc = encode_texts(tokenizer, data.text_clean, cfg["max_len"])
    batch_size = cfg["batch"] or auto_batch_size(device)
    pos_weight = (
        positive_class_weights(Y[fit_rows], "ratio", device) if condition == "weight" else None
    )
    t0 = time.time()
    best_f1, epochs_run, train_time = train_baseline_epochs(
        cfg,
        model,
        enc,
        Y,
        fit_rows,
        val_rows,
        batch_size,
        nn.BCEWithLogitsLoss(pos_weight=pos_weight),
        device,
        f"{config_plain} f{fold_id}",
    )
    s_val = predict_probabilities(model, enc, val_rows, batch_size * 2, device) - 0.5
    thresholds = np.array(
        [f1_optimal_threshold(Y[val_rows, j].astype(int), s_val[:, j]) for j in range(len(LABELS))]
    )
    s_test = predict_probabilities(model, enc, test_rows, batch_size * 2, device) - 0.5
    info = {
        "n_train": len(fit_rows),
        "device": device,
        "train_time": train_time,
        "t0": t0,
        "best_f1": best_f1,
        "Y_test": Y[test_rows].astype(int),
        "override": {
            "model": model_name,
            "batch": batch_size,
            "epochs_run": epochs_run,
            "lr": cfg["lr"],
        },
    }
    rows = baseline_result_rows(
        root,
        data,
        condition,
        fold_id,
        test_rows,
        [(False, s_test), (True, s_test - thresholds[None, :])],
        thresholds,
        info,
    )
    append_rows(root / "results" / csv_name, rows)
    for row in rows:
        print_run(row)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

## Revised Protocol Defaults

Default settings of the revised protocol. Each configuration overrides some of them.

In [ ]:
TRANSFORMER_DEFAULTS = {
    "model": "indolem/indobertweet-base-uncased",
    "text": "bert",
    "max_len": 128,
    "lr": 2e-5,
    "epochs": 6,
    "patience": 2,
    "batch": 16,
    "pos_weight": "sqrt10",
    "loss": "bce",
    "seed": RANDOM_STATE,
    "warmup": 0.1,
    "weight_decay": 0.01,
}

## Fine-Tune With Epoch Selection

Fine-tunes and keeps the epoch with the best inner validation macro F1 after threshold tuning. Returns the record of the best epoch, the training history and the training time, and loads the weights of the best epoch into the model.

In [ ]:
def fine_tune_with_epoch_selection(cfg, model, enc, Y, fit_rows, val_rows, device, log, tag):
    Y_tensor = torch.from_numpy(Y)
    if cfg["loss"] == "asl":
        loss_fn = asymmetric_loss
    else:
        loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=positive_class_weights(Y[fit_rows], cfg["pos_weight"], device)
        )
    tools = make_training_tools(
        model,
        len(fit_rows),
        cfg["batch"],
        cfg["epochs"],
        cfg["lr"],
        cfg["weight_decay"],
        cfg["warmup"],
        device,
    )
    t0 = time.time()
    best, history, bad_epochs = None, [], 0
    rng = np.random.default_rng(cfg["seed"])
    y_val = Y[val_rows].astype(int)
    for epoch in range(cfg["epochs"]):
        total_loss = train_one_epoch(
            model, enc, Y_tensor, fit_rows, rng, cfg["batch"], loss_fn, tools, device
        )
        p_val = predict_probabilities(model, enc, val_rows, cfg["batch"] * 4, device)
        thresholds = tune_probability_thresholds(y_val, p_val)
        f1_tuned, f1_default = macro_f1_at(y_val, p_val, thresholds), macro_f1_at(y_val, p_val, 0.5)
        history.append(
            {
                "epoch": epoch + 1,
                "loss": round(total_loss / math.ceil(len(fit_rows) / cfg["batch"]), 4),
                "inner_f1_tuned": round(f1_tuned, 4),
                "inner_f1_05": round(f1_default, 4),
            }
        )
        log(
            f"  {tag} epoch {epoch + 1}: inner macro-F1 tuned={f1_tuned:.4f} "
            f"(0.5: {f1_default:.4f}) {(time.time() - t0) / 60:.1f} min"
        )
        if best is None or f1_tuned > best["f"] + 1e-4:
            best, bad_epochs = {
                "f": f1_tuned,
                "epoch": epoch + 1,
                "thr": thresholds,
                "pv": p_val,
                "state": copy_model_state(model),
            }, 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                break
    model.load_state_dict(best["state"])
    return best, history, time.time() - t0

## Revised Result Rows

Builds one row with threshold 0.5 and one row with the tuned thresholds of the best epoch.

In [ ]:
def revised_result_rows(name, cfg, fold_id, best, history, p_test, Y_test, info):
    rows = []
    for tuned, thresholds in [(False, np.full(len(LABELS), 0.5)), (True, best["thr"])]:
        row = {
            "config": f"tf__{name}" + ("__tuned" if tuned else ""),
            "algo": "tf",
            "strategy": "mo",
            "condition": name,
            "features": cfg["text"],
            "tuned": tuned,
            "fold": fold_id,
            "n_train": info["n_train"],
            "n_test": len(Y_test),
            "device": info["device"],
            "train_time_s": round(info["train_time"], 1),
            "tune_time_s": 0.0,
            "total_time_s": round(time.time() - info["t0"], 1),
            "n_synthetic": 0,
            "override": json.dumps({k: cfg[k] for k in sorted(cfg)}),
            "thresholds": json.dumps([round(float(x), 4) for x in thresholds]) if tuned else "",
            "inner_log": json.dumps(history),
            "inner_macro_f1": round(best["f"], 4),
            "best_epoch": best["epoch"],
        }
        row.update(evaluate_predictions(Y_test, (p_test > thresholds[None, :]).astype(int)))
        rows.append(row)
    return rows

## Run Transformer Fold

Runs one configuration of the revised protocol on one outer fold. Appends the two result rows and stores the inner validation and test probabilities.

In [ ]:
def run_transformer_fold(
    root, name, cfg, data, folds, fold_id, device, done, csv_name=EXTENDED_CSV, log=print
):
    c = {**TRANSFORMER_DEFAULTS, **cfg}
    if (f"tf__{name}__tuned", fold_id) in done:
        log(f"skip tf__{name} f{fold_id} (already done)")
        return
    set_global_seed(c["seed"])
    train_rows, test_rows = fold_rows(data, folds[fold_id])
    Y = data[LABELS].to_numpy(dtype=np.float32)
    i_fit, i_val = inner_validation_split(Y[train_rows].astype(int))
    fit_rows, val_rows = train_rows[i_fit], train_rows[i_val]
    model_path = c["model"].replace("{fold}", str(fold_id))
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=len(LABELS), problem_type="multi_label_classification"
    ).to(device)
    texts = (
        data["Tweet"].map(normalize_for_transformer) if c["text"] == "bert" else data["text_clean"]
    )
    enc = encode_texts(tokenizer, texts, c["max_len"])
    best, history, train_time = fine_tune_with_epoch_selection(
        c, model, enc, Y, fit_rows, val_rows, device, log, f"tf__{name} f{fold_id}"
    )
    t0 = time.time() - train_time
    p_test = predict_probabilities(model, enc, test_rows, c["batch"] * 4, device)
    ids = data["id"].to_numpy()
    np.savez_compressed(
        root / "results" / "preds_ext" / f"tf__{name}__f{fold_id}.npz",
        test_ids=ids[test_rows],
        test_prob=p_test.astype(np.float32),
        inner_ids=ids[val_rows],
        inner_prob=best["pv"].astype(np.float32),
        thresholds=best["thr"].astype(np.float32),
    )
    info = {"n_train": len(fit_rows), "device": device, "train_time": train_time, "t0": t0}
    rows = revised_result_rows(
        name, c, fold_id, best, history, p_test, Y[test_rows].astype(int), info
    )
    append_rows(root / "results" / csv_name, rows)
    log(
        f"tf__{name} f{fold_id}: best epoch {best['epoch']}, inner {best['f']:.4f}, "
        f"test macro-F1 tuned={rows[1]['macro_f1']:.4f} (0.5: {rows[0]['macro_f1']:.4f}), "
        f"{train_time / 60:.1f} min"
    )
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

## Run Transformer Configurations

Runs every configuration of the revised protocol on every fold, skipping the folds that are already done.

In [ ]:
def run_transformer_configs(root, configs, folds_to_run=range(5), csv_name=EXTENDED_CSV, log=print):
    data, folds = load_inputs(root)
    out = root / "results" / csv_name
    for name, cfg in configs.items():
        for k in folds_to_run:
            run_transformer_fold(
                root, name, cfg, data, folds, k, DEVICE, completed_runs(out), csv_name, log
            )

## Ensemble One Fold

Averages the stored probabilities of the members on one fold and tunes the thresholds on the pooled inner validation data. Returns the result rows.

In [ ]:
def ensemble_fold(root, members, name, k, Y_by_id, out):
    preds_dir = root / "results" / "preds_ext"
    zs = [np.load(preds_dir / f"tf__{m}__f{k}.npz") for m in members]
    assert all(
        (z["inner_ids"] == zs[0]["inner_ids"]).all() and (z["test_ids"] == zs[0]["test_ids"]).all()
        for z in zs
    )
    p_val = np.mean([z["inner_prob"] for z in zs], 0)
    p_test = np.mean([z["test_prob"] for z in zs], 0)
    y_val = Y_by_id.loc[zs[0]["inner_ids"]].to_numpy(int)
    y_test = Y_by_id.loc[zs[0]["test_ids"]].to_numpy(int)
    thresholds = tune_probability_thresholds(y_val, p_val)
    np.savez_compressed(
        preds_dir / f"ens__{name}__f{k}.npz",
        test_ids=zs[0]["test_ids"],
        test_prob=p_test.astype(np.float32),
        inner_ids=zs[0]["inner_ids"],
        inner_prob=p_val.astype(np.float32),
        thresholds=thresholds.astype(np.float32),
    )
    previous = pd.read_csv(out)
    member_rows = previous.config.isin([f"tf__{m}__tuned" for m in members]) & (previous.fold == k)
    rows = []
    for tuned, t in [(False, np.full(len(LABELS), 0.5)), (True, thresholds)]:
        row = {
            "config": f"ens__{name}" + ("__tuned" if tuned else ""),
            "algo": "ens",
            "strategy": "mo",
            "condition": name,
            "features": "+".join(members),
            "tuned": tuned,
            "fold": k,
            "n_test": len(y_test),
            "train_time_s": float(previous[member_rows].train_time_s.sum()),
            "inner_macro_f1": round(macro_f1_at(y_val, p_val, thresholds), 4),
            "thresholds": json.dumps([round(float(x), 4) for x in t]) if tuned else "",
        }
        row.update(evaluate_predictions(y_test, (p_test > t[None, :]).astype(int)))
        rows.append(row)
    return rows

## Ensemble Transformers

Builds an ensemble on every fold whose members are complete and appends the result rows.

In [ ]:
def ensemble_transformers(root, members, name, csv_name=EXTENDED_CSV, log=print):
    data, _ = load_inputs(root)
    Y_by_id = data.set_index("id")[LABELS]
    out = root / "results" / csv_name
    done = completed_runs(out)
    for k in range(5):
        if (f"ens__{name}__tuned", k) in done:
            continue
        if not all(
            (root / "results" / "preds_ext" / f"tf__{m}__f{k}.npz").exists() for m in members
        ):
            log(f"ens__{name} f{k}: members incomplete, skipped")
            continue
        rows = ensemble_fold(root, members, name, k, Y_by_id, out)
        append_rows(out, rows)
        log(
            f"ens__{name} f{k}: test macro-F1 tuned={rows[1]['macro_f1']:.4f}, "
            f"inner={rows[1]['inner_macro_f1']:.4f}"
        )

## Masked Language Modelling Epoch

One epoch of masked language modelling, used by task-adaptive pretraining. Returns the mean loss.

In [ ]:
def mlm_epoch(model, features, collator, batch, generator, tools):
    optimizer, scheduler, scaler = tools
    model.train()
    order = torch.randperm(len(features), generator=generator).tolist()
    total, n_batches = 0.0, 0
    for start in range(0, len(order), batch):
        b = {
            k: v.to(DEVICE)
            for k, v in collator([features[i] for i in order[start : start + batch]]).items()
        }
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            loss = model(**b).loss
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += float(loss.detach())
        n_batches += 1
    return total / n_batches

## Task-Adaptive Pretraining

Continues masked language modelling (Gururangan et al., 2020) on the texts of the outer training fold only, without labels or test texts. It is kept for completeness and was not used in the reported results.

In [ ]:
def task_adaptive_pretraining(
    root, base_model, fold_id, out_dir, epochs=20, lr=5e-5, batch=32, max_len=128, log=print
):
    out_dir = Path(out_dir)
    if (out_dir / "config.json").exists():
        log(f"TAPT f{fold_id} already exists: {out_dir}")
        return out_dir
    set_global_seed(RANDOM_STATE)
    data, folds = load_inputs(root)
    train_rows, _ = fold_rows(data, folds[fold_id])
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForMaskedLM.from_pretrained(base_model).to(DEVICE)
    texts = data["Tweet"].iloc[train_rows].map(normalize_for_transformer).tolist()
    features = [
        {"input_ids": ids}
        for ids in tokenizer(texts, truncation=True, max_length=max_len)["input_ids"]
    ]
    collator = DataCollatorForLanguageModeling(tokenizer, mlm_probability=0.15)
    tools = make_training_tools(model, len(features), batch, epochs, lr, 0.01, 0.06, DEVICE)
    generator = torch.Generator().manual_seed(RANDOM_STATE)
    t0 = time.time()
    for epoch in range(epochs):
        loss = mlm_epoch(model, features, collator, batch, generator, tools)
        log(
            f"  TAPT f{fold_id} epoch {epoch + 1}/{epochs}: MLM loss {loss:.4f} "
            f"({(time.time() - t0) / 60:.1f} min)"
        )
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out_dir

# Threshold Strategies

Per-label thresholds tuned on about 1,500 inner validation tweets can overfit, especially for rare labels. Four strategies are compared for the transformers. The first keeps the default threshold of 0.5. The second uses the F1 optimal thresholds on the inner validation data. The third moves the tuned thresholds halfway towards 0.5. The fourth averages the tuned thresholds over 30 bootstrap samples of the inner validation data.

Each strategy is estimated inside the inner validation data. Thresholds are chosen on one half and scored on the other half, and this two-fold split is repeated five times. Only this estimate is used to choose a strategy, and the test score is reported afterwards.

## Bootstrap Mean Thresholds

Averages the tuned thresholds over bootstrap samples of the inner validation data.

In [ ]:
def bootstrap_mean_thresholds(Y, P, n_boot=30, seed=0):
    rng = np.random.default_rng(seed)
    n = len(Y)
    samples = (rng.integers(0, n, n) for _ in range(n_boot))
    return np.mean([tune_probability_thresholds(Y[i], P[i]) for i in samples], 0)

## Strategy Table

The four strategies as functions of the inner validation labels and probabilities.

In [ ]:
def fixed_thresholds(Y, P):
    return np.full(Y.shape[1], 0.5)


def shrunk_thresholds(Y, P):
    return (tune_probability_thresholds(Y, P) + 0.5) / 2


THRESHOLD_STRATEGIES = {
    "fixed_0.5": fixed_thresholds,
    "tuned": tune_probability_thresholds,
    "shrunk": shrunk_thresholds,
    "bootstrap_mean": bootstrap_mean_thresholds,
}

## Evaluate Threshold Strategies

Evaluates every strategy on one fold. inner_2fold is the honest estimate from repeated two-fold splits inside the inner validation data, and only this value may be used to choose a strategy. The test score uses thresholds chosen on all inner validation data.

In [ ]:
def evaluate_threshold_strategies(Y_by_id, npz_path, reps=5, seed=0):
    z = np.load(npz_path)
    y_val, p_val = Y_by_id.loc[z["inner_ids"]].to_numpy(int), z["inner_prob"]
    y_test, p_test = Y_by_id.loc[z["test_ids"]].to_numpy(int), z["test_prob"]
    rng = np.random.default_rng(seed)
    splits = []
    for _ in range(reps):
        idx = rng.permutation(len(y_val))
        a, b = idx[: len(idx) // 2], idx[len(idx) // 2 :]
        splits += [(a, b), (b, a)]
    out = {}
    for strategy, choose in THRESHOLD_STRATEGIES.items():
        t_full = choose(y_val, p_val)
        inner = np.mean(
            [macro_f1_at(y_val[B], p_val[B], choose(y_val[A], p_val[A])) for A, B in splits]
        )
        out[strategy] = {
            "inner_2fold": inner,
            "test_macro_f1": macro_f1_at(y_test, p_test, t_full),
            "thresholds": t_full,
            "test_ids": z["test_ids"],
            "test_prob": p_test,
        }
    return out

# Cross-Corpus Helpers

These functions are used in the cross-corpus evaluation. The IndoToxic2024 annotations are aggregated by majority vote for each label. Texts with tied votes and texts that most annotators marked as noise or spam are removed.

## Cross-Corpus Settings

The label mapping and the models that are scored on the 2024 corpus. The ensembles use threshold 0.5, as chosen by the threshold strategy evaluation.

In [ ]:
CROSS_CORPUS_FILE = EXTERNAL / "IndoDiscourse__indotoxic2024_annotated_data_v2_final.csv"
CROSS_LABEL_PAIRS = [
    ("HS", "toxicity"),
    ("HS", "identity_attack"),
    ("Abusive", "profanity_obscenity"),
]
CROSS_CLASSICAL = [
    ("svm", "br", "word", "none"),
    ("svm", "cc", "wordchar", "none"),
    ("cnb", "br", "word", "weight"),
]
CROSS_TRANSFORMERS = {
    "indobertweet_v2": {"model": "indolem/indobertweet-base-uncased"},
    "indobert_p1_v2": {"model": "indobenchmark/indobert-base-p1"},
    "indobertweet_v2_e10": {"model": "indolem/indobertweet-base-uncased", "epochs": 10},
    "indobertweet_v2_e10_s1": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 1,
    },
    "indobertweet_v2_e10_s2": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 2,
    },
    "indobert_p1_v2_e10": {"model": "indobenchmark/indobert-base-p1", "epochs": 10},
}
FINAL_ENSEMBLE_NAME = "ensemble 3-seed IndoBERTweet + IndoBERT-p1, 10 epochs (final model)"
CROSS_ENSEMBLES = {
    "ensemble IndoBERTweet + IndoBERT-p1 (6 epochs)": ["indobertweet_v2", "indobert_p1_v2"],
    FINAL_ENSEMBLE_NAME: [
        "indobertweet_v2_e10",
        "indobertweet_v2_e10_s1",
        "indobertweet_v2_e10_s2",
        "indobert_p1_v2_e10",
    ],
}

## Majority Vote

Majority vote over the annotators of one label. A tie gives a missing value.

In [ ]:
def majority_vote(cell):
    votes = [int(x) for x in ast.literal_eval(cell)] if isinstance(cell, str) else [int(cell)]
    positives = sum(votes)
    if 2 * positives == len(votes):
        return np.nan
    return int(2 * positives > len(votes))

## Load IndoToxic2024

Loads the corpus, aggregates the votes and removes noise or spam and empty texts.

In [ ]:
def load_indotoxic2024(path):
    df = pd.read_csv(path, keep_default_na=False)
    out = pd.DataFrame({"text_id": df.text_id, "text": df.text.astype(str)})
    for col in ["is_noise_or_spam_text", "toxicity", "identity_attack", "profanity_obscenity"]:
        out[col] = df[col].map(majority_vote)
    out["n_annotator"] = df.annotators_id.map(lambda s: len(ast.literal_eval(s)))
    keep = (out.is_noise_or_spam_text != 1) & (out.text.str.strip().str.len() > 0)
    return out[keep].reset_index(drop=True)

## Binary Metrics

F1, precision and recall of the positive class, plus the prevalence and the share of positive predictions.

In [ ]:
def binary_metrics(y, p):
    return {
        "f1_pos": f1_score(y, p, zero_division=0),
        "precision": precision_score(y, p, zero_division=0),
        "recall": recall_score(y, p, zero_division=0),
        "macro_f1_binary": f1_score(y, p, average="macro", zero_division=0),
        "prevalence": float(np.mean(y)),
        "predicted_positive_rate": float(np.mean(p)),
        "n": int(len(y)),
    }

## Score Cross-Corpus Predictions

Scores the predictions of one model for every label pair of the mapping, using only the texts with a rated label.

In [ ]:
def score_cross_corpus(corpus, pred, model_name):
    rows = []
    for source, target in CROSS_LABEL_PAIRS:
        rated = corpus[target].notna().to_numpy()
        y_true = corpus.loc[rated, target].astype(int).to_numpy()
        rows.append(
            {
                "model": model_name,
                "source_label": source,
                "target_label": target,
                **binary_metrics(y_true, pred[rated, LABEL_INDEX[source]]),
            }
        )
    return rows

## Train Classical Model on All Data

Trains on all Ibrohim and Budi data with hyperparameters and thresholds from inner validation data, then predicts the 2024 corpus.

In [ ]:
def train_classical_full(
    root, corpus, algo="svm", strategy="br", features="word", condition="none"
):
    data, _ = load_inputs(root)
    Y = label_matrix(data)
    override, thresholds, _ = select_on_inner_validation(
        algo, strategy, condition, data.text_clean.reset_index(drop=True), Y, features
    )
    vectorizers = fit_features(data.text_clean, features)
    model = fit_multilabel(
        strategy,
        algo,
        transform_features(vectorizers, data.text_clean),
        Y,
        condition == "weight",
        override,
    )
    X_corpus = transform_features(vectorizers, corpus.text_clean)
    return (decision_scores(model, X_corpus, thresholds) > 0).astype(int)

## Train Transformer on All Data

Trains a transformer on all Ibrohim and Budi data, with 15 percent held out for epoch selection, and stores its probabilities on the 2024 corpus. A stored result is reused.

In [ ]:
def train_transformer_full(root, name, cfg, corpus, log=print):
    out = root / "results" / "crosscorpus" / f"tf__{name}.npz"
    if out.exists():
        log(f"cross-corpus {name}: already done")
        return np.load(out)["prob"]
    out.parent.mkdir(parents=True, exist_ok=True)
    c = {**TRANSFORMER_DEFAULTS, **cfg}
    set_global_seed(c["seed"])
    data, _ = load_inputs(root)
    Y = data[LABELS].to_numpy(dtype=np.float32)
    i_fit, i_val = inner_validation_split(Y.astype(int))
    tokenizer = AutoTokenizer.from_pretrained(c["model"])
    model = AutoModelForSequenceClassification.from_pretrained(
        c["model"], num_labels=len(LABELS), problem_type="multi_label_classification"
    ).to(DEVICE)
    texts = list(data["Tweet"].map(normalize_for_transformer)) + list(
        corpus["text"].map(normalize_for_transformer)
    )
    enc = encode_texts(tokenizer, texts, c["max_len"])
    Y_all = np.vstack(
        [Y, np.zeros((len(corpus), len(LABELS)), np.float32)]
    )  # 2024 rows are never trained on
    best, _, train_time = fine_tune_with_epoch_selection(
        c, model, enc, Y_all, i_fit, i_val, DEVICE, log, f"cross-corpus {name}"
    )
    prob = predict_probabilities(
        model, enc, np.arange(len(data), len(texts)), c["batch"] * 4, DEVICE
    )
    np.savez_compressed(
        out,
        prob=prob.astype(np.float32),
        text_id=corpus.text_id.to_numpy().astype(str),
        best_epoch=best["epoch"],
        train_time_s=train_time,
    )
    log(f"cross-corpus {name}: best epoch {best['epoch']}, {train_time / 60:.1f} min")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return prob

## Out-of-Vocabulary Analysis

Share of tokens and of word types in the 2024 corpus that never occur in the Ibrohim and Budi data, and the most frequent of those words.

In [ ]:
def oov_analysis(root, corpus, top=30):
    data, _ = load_inputs(root)
    vocabulary = set(" ".join(data.text_clean).split())
    tokens = pd.Series(" ".join(corpus.text_clean).split())
    oov = tokens[~tokens.isin(vocabulary)]
    summary = {
        "oov_token_rate": len(oov) / max(len(tokens), 1),
        "oov_type_rate": oov.nunique() / max(tokens.nunique(), 1),
        "n_token": len(tokens),
    }
    return summary, oov.value_counts().head(top).rename_axis("token").reset_index(name="frequency")

## Prepare Cross-Corpus Data

Loads the 2024 corpus and applies the same preprocessing as the classical models.

In [ ]:
def prepare_cross_corpus(root):
    corpus = load_indotoxic2024(root / "data" / "external" / CROSS_CORPUS_FILE.name)
    preprocess = build_preprocessor(
        load_slang_dictionary(root / "data" / "raw" / "new_kamusalay.csv")
    )
    corpus["text_clean"] = corpus.text.map(preprocess)
    return corpus

## Run Cross-Corpus Evaluation

Scores the classical models, the transformers and the ensembles on the 2024 corpus and writes the out-of-vocabulary files.

In [ ]:
def run_cross_corpus(root, run_transformers=True, log=print):
    corpus = prepare_cross_corpus(root)
    log(
        f"2024 corpus after aggregation: {len(corpus)} texts | "
        f"positive rate toxicity {corpus.toxicity.mean():.3f}, "
        f"identity_attack {corpus.identity_attack.mean():.3f}, "
        f"profanity {corpus.profanity_obscenity.mean():.3f}"
    )
    rows = []
    for algo, strategy, features, condition in CROSS_CLASSICAL:
        pred = train_classical_full(root, corpus, algo, strategy, features, condition)
        rows += score_cross_corpus(
            corpus, pred, f"{algo}__{strategy}__{condition}__{features}__tuned"
        )
    probs = {}
    for name, cfg in CROSS_TRANSFORMERS.items():
        if run_transformers or (root / "results" / "crosscorpus" / f"tf__{name}.npz").exists():
            probs[name] = train_transformer_full(root, name, cfg, corpus, log)
    for name, p in probs.items():
        rows += score_cross_corpus(corpus, (p > 0.5).astype(int), f"tf__{name} (threshold 0.5)")
    for ensemble_name, members in CROSS_ENSEMBLES.items():
        if all(m in probs for m in members):
            rows += score_cross_corpus(
                corpus, (np.mean([probs[m] for m in members], 0) > 0.5).astype(int), ensemble_name
            )
    oov_summary, oov_top = oov_analysis(root, corpus)
    pd.DataFrame([oov_summary]).to_csv(root / "results" / "crosscorpus_oov.csv", index=False)
    oov_top.to_csv(root / "results" / "crosscorpus_top_oov.csv", index=False)
    log(
        f"OOV tokens {oov_summary['oov_token_rate']:.3f}, "
        f"OOV types {oov_summary['oov_type_rate']:.3f}"
    )
    return corpus, pd.DataFrame(rows), oov_summary, oov_top

# Sanity Checks

These small checks replace unit tests and run before any experiment.

## Deterministic TF-IDF Check

Ties are broken alphabetically, whatever the order of the documents.

In [ ]:
docs = ["bb aa cc", "cc bb aa", "dd ee", "ee dd", "zz yy"]
check_params = {"ngram_range": (1, 1), "max_features": 3, "sublinear_tf": True, "analyzer": "word"}
vocab_a = fit_deterministic_tfidf(docs, **check_params).get_feature_names_out()
vocab_b = fit_deterministic_tfidf(docs[::-1], **check_params).get_feature_names_out()
assert list(vocab_a) == ["aa", "bb", "cc"] == list(vocab_b)

## Hierarchy Check

The repair and the post-processing both return label sets that follow the annotation rules.

In [ ]:
y_check = np.zeros(len(LABELS), dtype=int)
y_check[LABEL_INDEX["HS_Gender"]] = 1
s_check = np.zeros(len(LABELS))
s_check[LABEL_INDEX["HS_Group"]], s_check[LABEL_INDEX["HS_Strong"]] = 2, 3
repaired_check = repair_synthetic_labelset(y_check, s_check, np.zeros(len(LABELS)))
assert count_hierarchy_violations(repaired_check[None, :])["any_violation_rows"] == 0
S_check = np.random.default_rng(1).normal(size=(500, len(LABELS)))
for mode in ("topdown", "bottomup"):
    assert (
        count_hierarchy_violations(enforce_label_hierarchy(S_check, mode))["any_violation_rows"]
        == 0
    )

## Threshold Check

A tuned threshold is never worse than the default threshold on the same data, and the fast threshold search gives the same answer as a plain loop over the same candidates.

In [ ]:
def reference_threshold(y, s):
    if y.sum() == 0:
        return 0.0
    candidates = np.unique(np.concatenate([np.quantile(s, np.linspace(0.01, 0.99, 99)), [0.0]]))
    best_f, best_t = -1.0, 0.0
    for t in candidates:
        f = f1_score(y, (s > t).astype(int), zero_division=0)
        if f > best_f + 1e-12 or (abs(f - best_f) <= 1e-12 and abs(t) < abs(best_t)):
            best_f, best_t = f, float(t)
    return best_t


rng_check = np.random.default_rng(0)
y_bin = (rng_check.random(500) < 0.1).astype(int)
s_bin = y_bin * 0.3 + rng_check.normal(0, 0.2, 500) - 0.4
tuned_f1 = f1_score(y_bin, (s_bin > f1_optimal_threshold(y_bin, s_bin)).astype(int))
assert tuned_f1 >= f1_score(y_bin, (s_bin > 0).astype(int))
for _ in range(100):
    y_rand = (rng_check.random(300) < rng_check.uniform(0.02, 0.5)).astype(int)
    s_rand = y_rand * rng_check.uniform(0, 1) + rng_check.normal(0, 0.3, 300)
    assert f1_optimal_threshold(y_rand, s_rand) == reference_threshold(y_rand, s_rand)

## Metric and Statistics Check

The fast macro F1 equals the scikit-learn value, and the statistical helpers return known results.

In [ ]:
Y_rand = (rng_check.random((400, len(LABELS))) < 0.2).astype(int)
P_rand = rng_check.random((400, len(LABELS)))
sklearn_macro = f1_score(Y_rand, (P_rand > 0.5).astype(int), average="macro", zero_division=0)
assert abs(macro_f1_at(Y_rand, P_rand, 0.5) - sklearn_macro) < 1e-12
np.testing.assert_allclose(holm_bonferroni([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
assert wilcoxon_signed_rank([2, 3, 4, 5], [1, 1, 1, 1])["rank_biserial"] == 1.0
print("All component checks passed.")

# Exploratory Data Analysis

All statistics in this part describe the raw data of 13,169 rows before deduplication. The statistics of the final data are in the deduplication step.

## Raw Data

The raw tweets, the slang dictionary and the abusive word lexicon are loaded. Abusive words are masked in the preview.

In [ ]:
raw = load_raw_dataset(RAW / "re_dataset.csv")
slang_dict = load_slang_dictionary(RAW / "new_kamusalay.csv")
abusive_lexicon = load_abusive_lexicon(RAW / "abusive.csv")
print("Raw rows:", len(raw))
print("Columns:", raw.columns.tolist())
print(
    "Slang dictionary entries:", len(slang_dict), "| abusive lexicon words:", len(abusive_lexicon)
)
raw.assign(Tweet=raw.Tweet.map(lambda t: mask_abusive_words(t, abusive_lexicon))).head()

## Label Frequency and Imbalance Ratio

Labels below 5 percent of the tweets are marked as minority labels. IRLbl and MeanIR follow Charte et al. (2015) and decide which labels MLSMOTE processes.

In [ ]:
Y_raw = label_matrix(raw)
label_counts = pd.DataFrame({"label": LABELS, "count": Y_raw.sum(axis=0)})
label_counts["percent"] = (label_counts["count"] / len(raw) * 100).round(2)
label_counts["minority_lt_5pct"] = label_counts["percent"] < 5
label_counts["IRLbl"] = irlbl(Y_raw).round(2)
MEAN_IR_RAW = mean_ir(Y_raw)
label_counts["IRLbl_gt_MeanIR"] = label_counts["IRLbl"] > MEAN_IR_RAW
label_counts = label_counts.sort_values("count", ascending=False).reset_index(drop=True)
label_counts.to_csv(RES / "eda_label_counts.csv", index=False)
print(f"MeanIR (raw data) = {MEAN_IR_RAW:.2f}")
label_counts

## Label Distribution Chart

The bar chart shows the label frequencies, with the minority labels in red.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = [
    "#c0392b" if is_minority else "#5d6d7e" for is_minority in label_counts["minority_lt_5pct"]
]
ax.bar(label_counts["label"], label_counts["count"], color=colors)
for i, (n, pct) in enumerate(zip(label_counts["count"], label_counts["percent"])):
    ax.text(i, n + 60, f"{pct:.1f}%", ha="center", fontsize=8)
ax.set_ylabel("Number of tweets")
ax.set_title("Label distribution (red = minority label, < 5%)")
plt.xticks(rotation=45, ha="right")
fig.savefig(FIG / "eda_label_distribution.png")
plt.show()

## Label Cardinality and Label Sets

Label cardinality is the mean number of labels per tweet, and label density divides it by the number of labels.

In [ ]:
labels_per_tweet = Y_raw.sum(axis=1)
cardinality = labels_per_tweet.mean()
labelsets = pd.Series(["".join(map(str, row)) for row in Y_raw])
multilabel_summary = pd.DataFrame(
    {
        "metric": [
            "n_tweets",
            "label_cardinality",
            "label_density",
            "n_unique_labelsets",
            "n_labelsets_seen_once",
        ],
        "value": [
            len(raw),
            round(cardinality, 3),
            round(cardinality / Y_raw.shape[1], 3),
            labelsets.nunique(),
            int((labelsets.value_counts() == 1).sum()),
        ],
    }
)
multilabel_summary.to_csv(RES / "eda_multilabel_summary.csv", index=False)
display(multilabel_summary)
labels_per_tweet_dist = (
    pd.Series(labels_per_tweet)
    .value_counts()
    .sort_index()
    .rename_axis("n_labels")
    .reset_index(name="n_tweets")
)
labels_per_tweet_dist.to_csv(RES / "eda_labels_per_tweet.csv", index=False)
labels_per_tweet_dist

## Label Co-occurrence

The heatmap counts how often each pair of labels appears in the same tweet.

In [ ]:
cooccurrence = pd.DataFrame(Y_raw.T @ Y_raw, index=LABELS, columns=LABELS)
cooccurrence.to_csv(RES / "eda_cooccurrence.csv")
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(
    cooccurrence, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, annot_kws={"size": 7}
)
ax.set_title("Label co-occurrence (diagonal = label frequency)")
fig.savefig(FIG / "eda_cooccurrence.png")
plt.show()

## Hierarchy Consistency

Counts the rows that break each annotation rule.

In [ ]:
violations_raw = count_hierarchy_violations(Y_raw)
pd.Series(violations_raw).to_frame("n_rows").to_csv(RES / "eda_hierarchy_check.csv")
print("Hierarchy violations:", violations_raw)
print(
    "Categories per HS tweet:",
    raw[raw.HS == 1][CATEGORY_LABELS].sum(axis=1).value_counts().sort_index().to_dict(),
)

## Tweet Length

Length of the cleaned text before slang normalisation, by HS class.

In [ ]:
cleaned_only = raw.Tweet.map(clean_text)
lengths = pd.DataFrame(
    {"HS": raw.HS, "n_char": cleaned_only.str.len(), "n_word": cleaned_only.str.split().str.len()}
)
length_stats = lengths.groupby("HS")[["n_char", "n_word"]].describe().round(1)
length_stats.to_csv(RES / "eda_length_by_hs.csv")
display(length_stats)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(data=lengths, x="n_word", hue="HS", bins=50, stat="density", common_norm=False, ax=ax)
ax.set_xlabel("Words per tweet")
ax.set_title("Tweet length distribution")
fig.savefig(FIG / "eda_length_hist.png")
plt.show()

# Preprocessing and Deduplication

The preprocessing for the classical models is applied without stemming and without stopword removal, and duplicates are removed.

## Apply Preprocessing

Every tweet is cleaned and normalised. Five examples are shown with abusive words masked.

In [ ]:
preprocess = build_preprocessor(slang_dict, use_stemming=False, use_stopwords=False)
raw["text_clean"] = raw.Tweet.map(preprocess)
example_idx = raw.sample(5, random_state=RANDOM_STATE).index
preprocessing_examples = pd.DataFrame(
    {
        "original": raw.loc[example_idx, "Tweet"],
        "cleaned": raw.loc[example_idx, "Tweet"].map(clean_text),
        "normalized": raw.loc[example_idx, "text_clean"],
    }
).apply(lambda col: col.map(lambda t: mask_abusive_words(t, abusive_lexicon)))
preprocessing_examples.to_csv(RES / "eda_preprocessing_examples.csv", index=False)
preprocessing_examples

## Deduplicate and Save

The value rows_final is the dataset size reported in the paper. The table at the end gives the label frequencies and imbalance ratios of the final data.

In [ ]:
data, dedup_report = deduplicate(raw[["Tweet", "text_clean"] + LABELS])
data.insert(0, "id", np.arange(len(data)))
pd.Series(dedup_report).to_frame("value").to_csv(RES / "eda_dedup_report.csv")
display(pd.Series(dedup_report).to_frame("value"))
assert count_hierarchy_violations(label_matrix(data))["any_violation_rows"] == 0

Y_final = label_matrix(data)
MEAN_IR_FINAL = mean_ir(Y_final)
final_counts = pd.DataFrame({"label": LABELS, "count": Y_final.sum(axis=0)})
final_counts["percent"] = (final_counts["count"] / len(data) * 100).round(2)
final_counts["IRLbl"] = irlbl(Y_final).round(2)
final_counts["IRLbl_gt_MeanIR"] = final_counts["IRLbl"] > MEAN_IR_FINAL
final_counts.to_csv(RES / "eda_label_counts_final.csv", index=False)
data.to_csv(PROC / "data_clean.csv", index=False)
print(
    "Saved:",
    PROC / "data_clean.csv",
    data.shape,
    f"| MeanIR (final data) = {MEAN_IR_FINAL:.2f}",
    f"| label cardinality = {Y_final.sum(axis=1).mean():.3f}",
)
final_counts

# Cross-Validation Folds

The folds come from MultilabelStratifiedKFold, the iterative stratification of Sechidis et al. (2011), with five folds, shuffling and random state 42. The file results/folds.json stores row ids rather than positions, so the folds stay valid if the order of the data file changes.

## Create or Load Folds

If folds.json exists it is loaded and no new folds are created. Delete the file only after the team agrees. The check confirms that every id is in exactly one test fold and that training and test ids never overlap.

In [ ]:
FOLDS_PATH = RES / "folds.json"
if FOLDS_PATH.exists():
    folds = json.loads(FOLDS_PATH.read_text())
    print("folds.json exists and was loaded (not recreated).")
else:
    ids = data["id"].to_numpy()
    splitter = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    folds = [
        {"fold": f, "train_ids": ids[tr].tolist(), "test_ids": ids[te].tolist()}
        for f, (tr, te) in enumerate(splitter.split(np.zeros(len(data)), label_matrix(data)))
    ]
    FOLDS_PATH.write_text(json.dumps(folds))
    print("folds.json created.")

all_test_ids = np.concatenate([f["test_ids"] for f in folds])
assert len(all_test_ids) == len(set(all_test_ids)) == len(data)
for f in folds:
    assert not set(f["train_ids"]) & set(f["test_ids"])
    assert len(f["train_ids"]) + len(f["test_ids"]) == len(data)
print("Fold check passed. Test size per fold:", [len(f["test_ids"]) for f in folds])

data, folds = load_inputs(ROOT)  # read back exactly as the experiment runner reads it
Y = label_matrix(data)

## Label Distribution per Fold

Counts each label in each test fold. The last line gives how much the share of each label varies across folds, in percentage points.

In [ ]:
fold_label_rows = []
for f in folds:
    _, test_rows = fold_rows(data, f)
    row = {"fold": f["fold"], "n_test": len(test_rows)}
    row.update({label: int(Y[test_rows, j].sum()) for j, label in enumerate(LABELS)})
    fold_label_rows.append(row)
fold_label_dist = pd.DataFrame(fold_label_rows).set_index("fold")
fold_label_dist.to_csv(RES / "folds_label_distribution.csv")
display(fold_label_dist)
label_share = fold_label_dist[LABELS].div(fold_label_dist.n_test, axis=0) * 100
(label_share.max() - label_share.min()).round(2).to_frame("range_pp").T

# MLSMOTE Inspection

The original MLSMOTE is compared with MLSMOTE plus hierarchy repair on the training data of fold 0. The features and MLSMOTE are fitted on that training data only.

## Resample Fold 0

Both versions of MLSMOTE are run on the training data of fold 0.

In [ ]:
train_rows_f0, _ = fold_rows(data, folds[0])
texts_f0 = data.text_clean.iloc[train_rows_f0]
X_train_f0 = transform_features(fit_features(texts_f0), texts_f0)
Y_train_f0 = Y[train_rows_f0]
print("Training matrix, fold 0:", X_train_f0.shape)

mlsmote_runs = {}
for name, repair in [("mlsmote_original", False), ("mlsmote_repair", True)]:
    t0 = time.time()
    _, Y_resampled, summary = mlsmote(
        X_train_f0, Y_train_f0, k=5, threshold="strict", repair=repair
    )
    mlsmote_runs[name] = {"seconds": time.time() - t0, "summary": summary, "Y": Y_resampled}

mlsmote_summary = pd.DataFrame(
    {
        name: {
            "seconds": round(run["seconds"], 2),
            "n_train_before": len(train_rows_f0),
            "n_synthetic": run["summary"]["n_synthetic"],
            "n_repaired": run["summary"]["n_repaired"],
            "n_synthetic_violating_hierarchy": count_hierarchy_violations(
                run["Y"][len(train_rows_f0) :]
            )["any_violation_rows"],
            "processed_labels": ", ".join(LABELS[i] for i in run["summary"]["processed_labels"]),
        }
        for name, run in mlsmote_runs.items()
    }
).T
mlsmote_summary.to_csv(RES / "mlsmote_fold0_summary.csv")
print(
    "MeanIR (fold 0 training data):",
    round(mlsmote_runs["mlsmote_original"]["summary"]["mean_ir"], 2),
)
mlsmote_summary

## Label Counts Before and After

Label counts and imbalance ratios before and after both MLSMOTE versions.

In [ ]:
original_run, repair_run = mlsmote_runs["mlsmote_original"], mlsmote_runs["mlsmote_repair"]
mlsmote_label_counts = pd.DataFrame(
    [
        {
            "label": label,
            "count_before": int(Y_train_f0[:, j].sum()),
            "count_original": int(original_run["Y"][:, j].sum()),
            "count_repair": int(repair_run["Y"][:, j].sum()),
            "IRLbl_before": round(original_run["summary"]["irlbl_before"][j], 2),
            "IRLbl_original": round(original_run["summary"]["irlbl_after"][j], 2),
            "IRLbl_repair": round(repair_run["summary"]["irlbl_after"][j], 2),
        }
        for j, label in enumerate(LABELS)
    ]
)
mlsmote_label_counts.to_csv(RES / "mlsmote_fold0_label_counts.csv", index=False)
mlsmote_label_counts

## Minority Labels Chart

The minority labels before and after MLSMOTE with repair.

In [ ]:
to_plot = mlsmote_label_counts.set_index("label").loc[
    MINORITY_LABELS, ["count_before", "count_repair"]
]
to_plot.columns = ["before MLSMOTE", "after MLSMOTE + repair"]
ax = to_plot.plot.bar(figsize=(7, 3.5), color=["#5d6d7e", "#c0392b"])
ax.set_title("Minority labels before and after MLSMOTE (fold 0, training data)")
ax.set_ylabel("Number of instances")
plt.xticks(rotation=0)
plt.savefig(FIG / "mlsmote_fold0_before_after.png")
plt.show()

## Design Decision

One pass of MLSMOTE, as in the paper, adds one synthetic instance per minority instance per label. The imbalance ratio drops, but labels such as HS_Gender remain minority labels. This is the behaviour of the original algorithm, not a bug.

In the original version, some synthetic instances break the label hierarchy, as shown in the column n_synthetic_violating_hierarchy. The main grid therefore uses the repair version, and the violations of the original version are reported as a finding in the repair ablation.

# Classical Model Grid

The grid crosses four algorithms, namely Complement Naive Bayes, LinearSVC, XGBoost and the MLP, with two multi-label strategies, Binary Relevance and Classifier Chains. Label Powerset is run only for the baseline comparison, with Complement Naive Bayes, LinearSVC and the random forest.

The imbalance treatments are none, class weighting and MLSMOTE with hierarchy repair. The original MLSMOTE without repair is run as an ablation. Every configuration is run with the default threshold and with inner validation tuning, on five folds of iterative stratification with seed 42. This gives 265 runs in total.

## Build the Grid

Builds the list of runs and shows the fixed hyperparameters and the tuning candidates.

In [ ]:
CLASSICAL_ALGOS = ["cnb", "svm"] + (["xgb", "mlp"] if RUN_GPU_MODELS else [])
main_grid = build_grid(
    CLASSICAL_ALGOS, ["br", "cc"], ["none", "weight", "mlsmote"], tuned=(False, True)
)
main_grid += build_grid(["cnb", "svm", "rf"], ["lp"], ["none"])
main_grid += build_grid(["cnb", "svm"], ["br"], ["mlsmote_raw"])
if not RUN_GPU_MODELS:
    print(
        "XGBoost and MLP are not run (RUN_GPU_MODELS=False). Their stored results are still used."
    )
print("Runs in grid:", len(main_grid))
display(
    pd.DataFrame({k: [v] for k, v in BASE_PARAMS.items()}).T.rename(
        columns={0: "fixed hyperparameters"}
    )
)
pd.Series(HYPERPARAMETER_GRID, name="tuning candidates").to_frame()

## Run the Grid

Runs that are already in experiments.csv are skipped.

In [ ]:
if RUN_GRID:
    run_grid(
        ROOT,
        main_grid,
        n_jobs_cpu=N_JOBS_CPU,
        csv_name=RESULTS_CSV,
        preds_subdir="preds",
        verbose=False,
    )
print("Rows in", RESULTS_CSV, ":", len(pd.read_csv(RES / RESULTS_CSV)))

# IndoBERTweet Baseline

IndoBERTweet, indolem/indobertweet-base-uncased, is fine-tuned for multi-label classification on the same folds with the baseline protocol. Fifteen percent of each training fold is used for early stopping and threshold selection. One model gives two rows, one with default thresholds and one with tuned thresholds. The weight condition means binary cross-entropy with positive class weights.

The transformer is not the focus of the study. It places the classical models against a reference upper bound. This part is skipped without a GPU.

## Run the Baseline Protocol

Runs both imbalance treatments on the five folds, skipping the folds that are already done.

In [ ]:
if RUN_BERT:
    done = completed_runs(RES / RESULTS_CSV)
    for condition in ["none", "weight"]:
        for k in range(5):
            run_bert_baseline_fold(
                ROOT, BERT_BASELINE_CONFIG, data, folds, k, condition, DEVICE, done
            )
else:
    print("IndoBERTweet baseline skipped (no GPU or RUN_BERT=False). Stored results are used.")

# Evaluation and Statistical Tests

Cross-validation folds are not independent because their training sets overlap, so folds are not used as Friedman blocks.

RQ2 on imbalance treatment and threshold tuning uses the Wilcoxon signed-rank test on the 12 per-label F1 scores, each the mean of five folds, and the corrected resampled t-test on per-fold macro F1. The Holm-Bonferroni correction is applied within each family of tests, and the matched-pairs rank-biserial correlation is the effect size.

RQ3 on the algorithms uses the Friedman test with the 12 labels as blocks, the Nemenyi post-hoc test and a critical difference diagram. One limitation is that the labels are not fully independent blocks, because of the annotation hierarchy.

## Load Results

Loads the complete runs of the main grid. The training and test sizes are used by the corrected resampled t-test.

In [ ]:
results = load_complete_runs(RES / RESULTS_CSV)
F1_COLUMNS = [f"f1__{label}" for label in LABELS]
ALGOS = [a for a in ["cnb", "svm", "xgb", "mlp"] if a in results.algo.unique()]
HAS_BERT = "bert" in results.algo.unique()
KEY = ["algo", "strategy", "condition", "tuning"]
N_TRAIN = int(results.n_train[results.condition == "none"].mean())
N_TEST = int(results.n_test.mean())
print(results.shape, results.config.nunique(), "configurations | IndoBERTweet present:", HAS_BERT)

## Per-Fold Macro F1

Returns the macro F1 of one configuration on each fold, in fold order.

In [ ]:
def fold_macro_f1(df, key_cols, key):
    mask = np.logical_and.reduce([(df[c] == v).to_numpy() for c, v in zip(key_cols, key)])
    return df[mask].sort_values("fold").macro_f1.to_numpy()

## Main Table

Mean and standard deviation over five folds for every configuration.

In [ ]:
MAIN_METRICS = [
    "macro_f1",
    "micro_f1",
    "minority_macro_f1",
    "jaccard_acc",
    "hamming_loss",
    "pred_hierarchy_violation_rate",
    "total_time_s",
]
grouped = results.groupby(KEY)[MAIN_METRICS]
config_mean, config_std = grouped.mean(), grouped.std()
main_table = (
    config_mean.round(4).astype(str) + " ± " + config_std.round(4).astype(str)
).reset_index()
main_table["algo"] = main_table.algo.map(ALGO_NAMES)
save_table(main_table, "table_main.csv", index=False, decimals=None)
save_table(config_mean, "table_main_mean.csv")
save_table(config_std, "table_main_std.csv")
config_mean_flat = config_mean.reset_index()
ranking = config_mean_flat.sort_values("macro_f1", ascending=False).assign(
    algo=lambda d: d.algo.map(ALGO_NAMES)
)
ranking.head(15).round(4)

## F1 per Label

F1 of each label for every configuration, and a table with the best configuration of each algorithm by mean macro F1.

In [ ]:
per_label = results.groupby(KEY)[F1_COLUMNS].mean()
per_label.columns = LABELS
save_table(per_label, "table_f1_per_label.csv")
best_per_algo = config_mean_flat.sort_values("macro_f1", ascending=False).groupby("algo").head(1)
best_per_algo_f1 = pd.DataFrame(
    [
        pd.Series(
            per_label.loc[(r.algo, r.strategy, r.condition, r.tuning)],
            name=f"{ALGO_NAMES[r.algo]} | {r.strategy.upper()} | {r.condition} | {r.tuning}",
        )
        for _, r in best_per_algo.iterrows()
    ]
)
save_table(best_per_algo_f1, "table_f1_per_label_best_per_algo.csv", decimals=3)

## Imbalance Treatment Chart

Macro F1 and minority macro F1 across imbalance treatments for Binary Relevance, with and without tuning.

In [ ]:
CONDITIONS_MAIN = ["none", "weight", "mlsmote"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey="row")
for i, metric in enumerate(["macro_f1", "minority_macro_f1"]):
    for j, tuning in enumerate(["fixed", "tuned"]):
        subset = config_mean_flat[
            (config_mean_flat.strategy == "br")
            & (config_mean_flat.tuning == tuning)
            & config_mean_flat.condition.isin(CONDITIONS_MAIN)
        ]
        pivot = subset.pivot(index="algo", columns="condition", values=metric).reindex(
            index=ALGOS, columns=CONDITIONS_MAIN
        )
        pivot.index = pivot.index.map(ALGO_NAMES)
        pivot.columns = [IMBALANCE_NAMES[c] for c in pivot.columns]
        pivot.plot.bar(
            ax=axes[i, j], color=["#95a5a6", "#2e86c1", "#c0392b"], width=0.75, legend=False
        )
        axes[i, j].set_title(
            f"{'Macro-F1' if i == 0 else 'Minority macro-F1'} | BR | {tuning}", fontsize=10
        )
        axes[i, j].set_xlabel("")
        axes[i, j].tick_params(axis="x", rotation=0)
handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    legend_labels,
    title="imbalance treatment",
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 1.09),
)
fig.tight_layout()
fig.savefig(FIG / "macro_f1_by_condition.png")
plt.show()

## F1 Heatmap

F1 of every label for the Binary Relevance and Classifier Chains configurations and the transformer.

In [ ]:
heatmap_data = per_label.reset_index()
heatmap_data = heatmap_data[
    heatmap_data.strategy.isin(["br", "cc", "mo"]) & heatmap_data.condition.isin(CONDITIONS_MAIN)
]
heatmap_data.index = [
    f"{ALGO_NAMES[a]} | {s.upper()} | {c} | {t}"
    for a, s, c, t in zip(
        heatmap_data.algo, heatmap_data.strategy, heatmap_data.condition, heatmap_data.tuning
    )
]
heatmap_data = heatmap_data[LABELS]
fig, ax = plt.subplots(figsize=(12, max(6, 0.28 * len(heatmap_data))))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "F1"},
    ax=ax,
    annot_kws={"size": 6},
)
ax.set_title("Per-label F1 (mean over 5 folds)")
fig.savefig(FIG / "heatmap_f1_per_label.png")
plt.show()

## RQ2 Imbalance Treatment and Threshold Tuning

The comparisons are paired within the same algorithm and strategy. The imbalance family compares none with weight, none with mlsmote and weight with mlsmote, all with fixed thresholds. The tuning family compares fixed with tuned for each imbalance treatment.

In [ ]:
rq2_rows = []
for algo in ALGOS:
    for strategy in ["br", "cc"]:
        pairs = [
            ((a, "fixed"), (b, "fixed"), "imbalance")
            for a, b in [("none", "weight"), ("none", "mlsmote"), ("weight", "mlsmote")]
        ]
        pairs += [((c, "fixed"), (c, "tuned"), "tuning") for c in CONDITIONS_MAIN]
        for A, B, family in pairs:
            key_a, key_b = (algo, strategy, *A), (algo, strategy, *B)
            if key_a not in per_label.index or key_b not in per_label.index:
                continue  # skip incomplete configurations
            comparison = paired_comparison(
                per_label.loc[key_a].to_numpy(),
                per_label.loc[key_b].to_numpy(),
                fold_macro_f1(results, KEY, key_a),
                fold_macro_f1(results, KEY, key_b),
                N_TRAIN,
                N_TEST,
            )
            rq2_rows.append(
                {
                    "family": family,
                    "algo": ALGO_NAMES[algo],
                    "strategy": strategy.upper(),
                    "A": "/".join(A),
                    "B": "/".join(B),
                    **comparison,
                }
            )
rq2 = add_holm_columns(pd.DataFrame(rq2_rows), ["wilcoxon_p", "corrected_t_p"], by="family")
rq2 = move_columns_last(rq2, EFFECT_SIZE_COLUMNS)
save_table(rq2, "stats_rq2.csv", index=False)

## RQ3 Differences Between Algorithms

The Friedman test, with the 12 labels as blocks, compares the algorithms under the same strategy, imbalance treatment and tuning. One extra comparison uses the best configuration of each algorithm by mean macro F1, including IndoBERTweet when it is present. That comparison is descriptive, because the configurations are selected on the same results.

In [ ]:
friedman_rows = []
for strategy in ["br", "cc"]:
    for condition in CONDITIONS_MAIN:
        for tuning in ["fixed", "tuned"]:
            available = [a for a in ALGOS if (a, strategy, condition, tuning) in per_label.index]
            if len(available) < 3:
                continue  # the Friedman test needs at least three algorithms
            M = pd.DataFrame(
                {
                    ALGO_NAMES[a]: per_label.loc[(a, strategy, condition, tuning)].to_numpy()
                    for a in available
                },
                index=LABELS,
            )
            out = friedman_nemenyi(M)
            friedman_rows.append(
                {
                    "strategy": strategy.upper(),
                    "condition": condition,
                    "tuning": tuning,
                    "friedman_chi2": out["chi2"],
                    "friedman_p": out["p"],
                    **{f"avg_rank_{k}": v for k, v in out["avg_rank"].items()},
                }
            )
            save_table(out["nemenyi"], f"stats_rq3_nemenyi_{strategy}_{condition}_{tuning}.csv")
friedman = pd.DataFrame(friedman_rows)
if len(friedman):
    friedman["friedman_p_holm"] = holm_bonferroni(friedman.friedman_p.to_numpy())
save_table(friedman, "stats_rq3_friedman.csv", index=False)

best_config = {}
for algo in ALGOS + (
    ["bert"] if HAS_BERT else []
):  # best configuration of each algorithm, plus IndoBERTweet
    r = (
        config_mean_flat[config_mean_flat.algo == algo]
        .sort_values("macro_f1", ascending=False)
        .iloc[0]
    )
    best_config[ALGO_NAMES[algo]] = (algo, r.strategy, r.condition, r.tuning)
M_best = pd.DataFrame(
    {name: per_label.loc[key].to_numpy() for name, key in best_config.items()}, index=LABELS
)
print("Best configuration per algorithm:", best_config)
if M_best.shape[1] >= 3:
    out_best = friedman_nemenyi(M_best)
    save_table(out_best["nemenyi"], "stats_rq3_nemenyi_best_per_algo.csv")
    print(
        f"Friedman (best configuration per algorithm): chi2={out_best['chi2']:.2f}, "
        f"p={out_best['p']:.4g}"
    )
friedman.round(4)

## Plot Critical Difference Diagram

Draws one critical difference diagram from a table of scores, with the average rank of each method.

In [ ]:
ALGO_COLORS = {
    "Complement NB": "#7f8c8d",
    "LinearSVC": "#2e86c1",
    "XGBoost": "#c0392b",
    "MLP": "#27ae60",
    "IndoBERTweet": "#8e44ad",
}


def plot_cd_diagram(ax, M, title):
    ranks = M.rank(axis=1, ascending=False).mean()
    nemenyi = skp.posthoc_nemenyi_friedman(M.to_numpy())
    nemenyi.index = nemenyi.columns = M.columns
    plt.sca(ax)
    skp.critical_difference_diagram(
        ranks,
        nemenyi,
        ax=ax,
        color_palette={k: ALGO_COLORS[k] for k in M.columns},
        label_fmt_left="{label} ({rank:.2f})",
        label_fmt_right="({rank:.2f}) {label}",
    )
    ax.set_title(title, fontsize=9)

## Critical Difference Diagrams

Diagrams for three Binary Relevance settings and for the best configuration of each algorithm.

In [ ]:
cd_panels = []
for condition, tuning in [("none", "fixed"), ("weight", "fixed"), ("none", "tuned")]:
    available = [a for a in ALGOS if (a, "br", condition, tuning) in per_label.index]
    if len(available) >= 3:
        M = pd.DataFrame(
            {
                ALGO_NAMES[a]: per_label.loc[(a, "br", condition, tuning)].to_numpy()
                for a in available
            },
            index=LABELS,
        )
        cd_panels.append((M, f"BR | {condition} | {tuning}"))
if M_best.shape[1] >= 3:
    cd_panels.append((M_best, "Best configuration per algorithm (descriptive)"))
if cd_panels:
    fig, axes = plt.subplots(len(cd_panels), 1, figsize=(7.5, 2 * len(cd_panels)), squeeze=False)
    for ax, (M, title) in zip(axes[:, 0], cd_panels):
        plot_cd_diagram(ax, M, title)
    plt.tight_layout()
    fig.savefig(FIG / "cd_diagram.png")
    plt.show()

## Baseline Comparison

The baseline of Ibrohim and Budi (2019) uses a random forest with Label Powerset and word unigrams and reaches an accuracy of 66.12 percent on 12 labels with 10 folds. Accuracy in that paper is the Jaccard index per document, their Equation 1. Three setup differences must be reported: five folds instead of ten, deduplicated data, and TF-IDF unigrams and bigrams.

In [ ]:
BASELINE_JACCARD_PCT = 66.12
lp_table = config_mean_flat[config_mean_flat.strategy == "lp"][
    ["algo", "jaccard_acc", "macro_f1", "minority_macro_f1"]
].copy()
lp_table["algo"] = lp_table.algo.map(ALGO_NAMES)
lp_table["diff_vs_baseline_pp"] = (lp_table.jaccard_acc * 100 - BASELINE_JACCARD_PCT).round(2)
best_jaccard = config_mean_flat.sort_values("jaccard_acc", ascending=False).head(8)[
    KEY + ["jaccard_acc", "macro_f1"]
]
best_jaccard["algo"] = best_jaccard.algo.map(ALGO_NAMES)
save_table(lp_table, "table_baseline_lp.csv", index=False)
save_table(best_jaccard, "table_best_jaccard.csv", index=False)
display(lp_table.round(4))
best_jaccard.round(4)

## Computation Time and Hierarchy Violations

Mean training time, tuning time and hierarchy violation rate of the predictions for every configuration and device.

In [ ]:
time_violation = results.groupby(KEY + ["device"])[
    ["train_time_s", "tune_time_s", "total_time_s", "pred_hierarchy_violation_rate"]
].mean()
save_table(time_violation, "table_time_violation.csv", decimals=3)

## MLSMOTE Repair Ablation

The original MLSMOTE against MLSMOTE with hierarchy repair, for Complement Naive Bayes and LinearSVC with Binary Relevance.

In [ ]:
ablation = results[
    (results.strategy == "br")
    & (results.tuning == "fixed")
    & results.condition.isin(["mlsmote", "mlsmote_raw"])
    & results.algo.isin(["cnb", "svm"])
]
ablation = ablation.groupby(["algo", "condition"])[
    [
        "macro_f1",
        "minority_macro_f1",
        "jaccard_acc",
        "pred_hierarchy_violation_rate",
        "n_synthetic",
        "n_repaired",
    ]
].mean()
save_table(ablation, "table_ablation_repair.csv")

## Key Numbers

A summary of the main numbers for the authors. These lines are not paper text.

In [ ]:
for metric in ["macro_f1", "minority_macro_f1", "jaccard_acc"]:
    r = config_mean_flat.sort_values(metric, ascending=False).iloc[0]
    print(
        f"- highest {metric}: {ALGO_NAMES[r.algo]} {r.strategy.upper()} {r.condition} "
        f"{r.tuning} = {r[metric]:.4f}"
    )
best_classical = (
    config_mean_flat[config_mean_flat.algo != "bert"]
    .sort_values("macro_f1", ascending=False)
    .iloc[0]
)
print(
    f"- best classical model: {ALGO_NAMES[best_classical.algo]} {best_classical.strategy.upper()} "
    f"{best_classical.condition} {best_classical.tuning} = {best_classical.macro_f1:.4f}"
)
for _, r in rq2.iterrows():
    verdict = "significant" if r.wilcoxon_p_holm < 0.05 else "not significant"
    print(
        f"- [{r.family}] {r.algo} {r.strategy}: {r.A} -> {r.B}: "
        f"delta macro-F1 {r.delta_macro_f1:+.4f}, delta minority-F1 {r.delta_minority_f1:+.4f}, "
        f"{r.n_labels_improved}/12 labels improved, Wilcoxon-Holm p={r.wilcoxon_p_holm:.3f} "
        f"({verdict}), rank-biserial r={r.rank_biserial:+.2f}"
    )

## Best Classical Model Against IndoBERTweet

The two best configurations by mean macro F1 are compared with the Wilcoxon signed-rank test on per-label F1 and the corrected resampled t-test on per-fold macro F1. Because both configurations are selected on the same results, this is a weak confirmatory test. It is reported next to the Nemenyi test, which is more conservative because it compares five methods at once.

In [ ]:
if HAS_BERT:
    best_transformer = (
        config_mean_flat[config_mean_flat.algo == "bert"]
        .sort_values("macro_f1", ascending=False)
        .iloc[0]
    )
    key_classical = (
        best_classical.algo,
        best_classical.strategy,
        best_classical.condition,
        best_classical.tuning,
    )
    key_transformer = (
        best_transformer.algo,
        best_transformer.strategy,
        best_transformer.condition,
        best_transformer.tuning,
    )
    f1_classical, f1_transformer = (
        per_label.loc[key_classical].to_numpy(),
        per_label.loc[key_transformer].to_numpy(),
    )
    wilcoxon = wilcoxon_signed_rank(f1_transformer, f1_classical)
    ttest = corrected_resampled_ttest(
        fold_macro_f1(results, KEY, key_transformer),
        fold_macro_f1(results, KEY, key_classical),
        N_TRAIN,
        N_TEST,
    )
    id_classical = config_id(*key_classical[:3], key_classical[3] == "tuned")
    id_transformer = config_id(*key_transformer[:3], key_transformer[3] == "tuned")
    classical_vs_transformer = pd.DataFrame(
        [
            {
                "classical_config": " | ".join(map(str, key_classical)),
                "transformer_config": " | ".join(map(str, key_transformer)),
                "macro_f1_classical": f1_classical.mean(),
                "macro_f1_transformer": f1_transformer.mean(),
                "delta_macro_f1": f1_transformer.mean() - f1_classical.mean(),
                "n_labels_transformer_better": int((f1_transformer > f1_classical).sum()),
                "wilcoxon_p": wilcoxon["p"],
                "corrected_t_p": ttest["p"],
                "train_time_classical_s": results[
                    results.config == id_classical
                ].train_time_s.mean(),
                "train_time_transformer_s": results[
                    results.config == id_transformer
                ].train_time_s.mean(),
                "wilcoxon_stat": wilcoxon["stat"],
                "rank_biserial": wilcoxon["rank_biserial"],
                "corrected_t": ttest["t"],
            }
        ]
    )
    display(save_table(classical_vs_transformer, "stats_light_vs_bert.csv", index=False).T)
else:
    print("No IndoBERTweet results yet.")

# Feature Inspection and Error Analysis

This part lists the most influential LinearSVC features for the minority labels and samples out-of-fold false negatives. The team fills in the cause column of the false negative file by hand, and existing notes are kept when this part is rerun. Abusive words are masked.

## Select Configuration

The best LinearSVC configuration with Binary Relevance by mean macro F1.

In [ ]:
svm_br_scores = (
    results[(results.algo == "svm") & (results.strategy == "br")]
    .groupby("config")
    .macro_f1.mean()
    .sort_values(ascending=False)
)
INSPECT_CONFIG = svm_br_scores.index[0]
INSPECT_CONDITION = INSPECT_CONFIG.split("__")[2]
print("Configuration inspected:", INSPECT_CONFIG)

## Most Influential Features

The 20 words with the largest LinearSVC coefficients for each minority label, from a model trained on the training data of fold 0.

In [ ]:
inspect_vectorizers = fit_features(texts_f0)
X_inspect, Y_inspect = transform_features(inspect_vectorizers, texts_f0), Y[train_rows_f0]
if INSPECT_CONDITION in RESAMPLING_CONDITIONS:
    X_inspect, Y_inspect, _ = resample(X_inspect, Y_inspect, INSPECT_CONDITION)
inspect_models = fit_binary_relevance(
    "svm", X_inspect, Y_inspect, weight=(INSPECT_CONDITION == "weight")
)
vocabulary = np.array(inspect_vectorizers[0].get_feature_names_out())
top_features = {}
for label in MINORITY_LABELS:
    coef = inspect_models[LABEL_INDEX[label]].coef_.ravel()
    top = np.argsort(coef)[::-1][:20]
    top_features[label] = [
        f"{mask_abusive_words(vocabulary[i], abusive_lexicon)} ({coef[i]:.2f})" for i in top
    ]
top_features = pd.DataFrame(top_features)
top_features.to_csv(RES / "explain_top_features_svm.csv", index=False)
top_features

## False Negatives

Ten out-of-fold false negatives are sampled for each minority label. Manual notes from an earlier run are kept, including the penyebab column of the previous notebook.

In [ ]:
oof_scores = np.zeros_like(Y, dtype=float)
row_of_id = pd.Series(np.arange(len(data)), index=data["id"])
for k in range(5):
    test_ids, fold_scores = load_fold_scores(INSPECT_CONFIG, k)
    oof_scores[row_of_id[test_ids].to_numpy()] = fold_scores
oof_pred = (oof_scores > 0).astype(int)

fn_rows, rng_fn = [], np.random.default_rng(RANDOM_STATE)
for label in MINORITY_LABELS:
    j = LABEL_INDEX[label]
    false_negatives = np.where((Y[:, j] == 1) & (oof_pred[:, j] == 0))[0]
    for i in rng_fn.choice(false_negatives, size=min(10, len(false_negatives)), replace=False):
        fn_rows.append(
            {
                "label": label,
                "id": int(data.id.iloc[i]),
                "score": round(oof_scores[i, j], 3),
                "true_labels": ", ".join(l for l, v in zip(LABELS, Y[i]) if v),
                "predicted_labels": ", ".join(l for l, v in zip(LABELS, oof_pred[i]) if v) or "-",
                "tweet_masked": mask_abusive_words(data.Tweet.iloc[i], abusive_lexicon),
                "cause": "",
            }
        )
false_negative_examples = pd.DataFrame(fn_rows)

fn_path = RES / "error_analysis_false_negatives.csv"
if fn_path.exists():
    previous = pd.read_csv(fn_path, keep_default_na=False)
    note_col = next((c for c in ["cause", "penyebab"] if c in previous.columns), None)
    if note_col is not None:
        notes = previous.set_index(["label", "id"])[note_col].astype(str)
        keys = list(zip(false_negative_examples.label, false_negative_examples.id))
        false_negative_examples["cause"] = [notes.get(key, "") for key in keys]
        print(f"Manual notes kept: {(false_negative_examples.cause.str.strip() != '').sum()}")
false_negative_examples.to_csv(fn_path, index=False)

error_counts = pd.DataFrame(
    {
        label: {
            "positives": int(Y[:, LABEL_INDEX[label]].sum()),
            "false_negatives": int(
                ((Y[:, LABEL_INDEX[label]] == 1) & (oof_pred[:, LABEL_INDEX[label]] == 0)).sum()
            ),
            "false_positives": int(
                ((Y[:, LABEL_INDEX[label]] == 0) & (oof_pred[:, LABEL_INDEX[label]] == 1)).sum()
            ),
        }
        for label in MINORITY_LABELS
    }
).T
error_counts.to_csv(RES / "error_analysis_counts.csv")
display(error_counts)
false_negative_examples.head(10)

## Missed Categories

When a category is missed, the table shows which category was predicted instead and how often HS itself was missed.

In [ ]:
category_confusion = {}
for label in ["HS_Gender", "HS_Physical", "HS_Race", "HS_Religion"]:
    j = LABEL_INDEX[label]
    missed = (Y[:, j] == 1) & (oof_pred[:, j] == 0)
    category_confusion[label] = {
        c: int(oof_pred[missed, LABEL_INDEX[c]].sum()) for c in CATEGORY_LABELS
    } | {
        "not_predicted_HS": int((oof_pred[missed, HS_INDEX] == 0).sum()),
        "total_FN": int(missed.sum()),
    }
category_confusion = pd.DataFrame(category_confusion).T
category_confusion.to_csv(RES / "error_analysis_category_confusion.csv")
category_confusion

# Follow-Up Experiments

These experiments answer reviewer style questions and test improvements from the literature. All runs go to results/experiments_ext.csv, with decision scores in results/preds_ext, so the main tables do not change.

The follow-up experiments ask whether post-processing with the annotation hierarchy improves the predictions without retraining, whether random oversampling with ML-ROS fails like MLSMOTE, whether character n-grams help the classical models, why oversampling fails, and whether the transformer can be improved with the original text, epoch selection, another model and ensembles. They end with the choice of the threshold strategy and the final model, and a comparison of the final model with the earlier models.

The same rule applies as before. Every choice of hyperparameters, thresholds, epochs, ensemble thresholds and final configuration uses only the inner validation data of the outer training fold. Every attempt is reported, including those that did not help.

## Follow-Up Settings

The transformer configurations and ensembles of the revised protocol, keyed by configuration name. An ensemble averages the member probabilities and tunes thresholds on the pooled inner validation data.

In [ ]:
EXTENDED_ALGOS_TO_RUN = ["cnb", "svm"] + (["xgb", "mlp"] if RUN_GPU_MODELS else [])
TRANSFORMER_CONFIGS = {
    "indobertweet_v2": {"model": "indolem/indobertweet-base-uncased"},
    "indobert_p1_v2": {"model": "indobenchmark/indobert-base-p1"},
    "indobertweet_v2_e10": {"model": "indolem/indobertweet-base-uncased", "epochs": 10},
    "indobertweet_v2_s1": {"model": "indolem/indobertweet-base-uncased", "seed": 1},
    "indobertweet_v2_s2": {"model": "indolem/indobertweet-base-uncased", "seed": 2},
    "indobertweet_v2_e10_s1": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 1,
    },
    "indobertweet_v2_e10_s2": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 2,
    },
    "indobert_p1_v2_e10": {"model": "indobenchmark/indobert-base-p1", "epochs": 10},
}
TRANSFORMER_ENSEMBLES = {
    "indobertweet+indobert_p1": ["indobertweet_v2", "indobert_p1_v2"],
    "indobertweet_x3": ["indobertweet_v2", "indobertweet_v2_s1", "indobertweet_v2_s2"],
    "indobertweet_x3+indobert_p1": [
        "indobertweet_v2",
        "indobertweet_v2_s1",
        "indobertweet_v2_s2",
        "indobert_p1_v2",
    ],
    "e10_indobertweet+indobert_p1": ["indobertweet_v2_e10", "indobert_p1_v2_e10"],
    "e10_indobertweet_x3": [
        "indobertweet_v2_e10",
        "indobertweet_v2_e10_s1",
        "indobertweet_v2_e10_s2",
    ],
    "e10_indobertweet_x3+indobert_p1": [
        "indobertweet_v2_e10",
        "indobertweet_v2_e10_s1",
        "indobertweet_v2_e10_s2",
        "indobert_p1_v2_e10",
    ],
}
Y_BY_ID = data.set_index("id")[LABELS]
print(
    "RUN_EXTENDED_GRID =",
    RUN_EXTENDED_GRID,
    "| RUN_TRANSFORMERS =",
    RUN_TRANSFORMERS,
    "| transformer configurations:",
    list(TRANSFORMER_CONFIGS),
)

## Load Extended Results

Reads the run log of the follow-up experiments, or an empty table when it does not exist yet.

In [ ]:
def load_extended():
    path = RES / EXTENDED_CSV
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

## Post-Processing Table

Binary Relevance predictions often break the annotation rules, for example when HS_Gender is predicted without HS. The rules come from the annotation guideline, not from the test data, so there is no leakage. The topdown variant was fixed as the main variant before the results were seen, and the bottomup variant is used for comparison. The function applies both to the stored scores of every Binary Relevance, Classifier Chains and transformer configuration, fold by fold. The variant column holds the imbalance treatment for the classical models and the configuration name for the transformers.

In [ ]:
def postprocessing_row(config, meta):
    per_mode = {mode: [] for mode in ["original", "topdown", "bottomup"]}
    for k in range(5):
        ids, S = load_fold_scores(config, k)
        Y_true = Y_BY_ID.loc[ids].to_numpy(int)
        per_mode["original"].append(evaluate_predictions(Y_true, (S > 0).astype(int)))
        for mode in ["topdown", "bottomup"]:
            per_mode[mode].append(evaluate_predictions(Y_true, enforce_label_hierarchy(S, mode)))
    row = {
        "config": config,
        "algo": meta.algo,
        "strategy": meta.strategy,
        "variant": meta.condition,
        "tuning": "tuned" if meta.tuned else "fixed",
    }
    f1 = {
        mode: np.mean([[d[f"f1__{label}"] for label in LABELS] for d in per_mode[mode]], 0)
        for mode in per_mode
    }
    for mode in per_mode:
        for metric in [
            "macro_f1",
            "minority_macro_f1",
            "jaccard_acc",
            "subset_acc",
            "pred_hierarchy_violation_rate",
        ]:
            row[f"{metric}__{mode}"] = np.mean([d[metric] for d in per_mode[mode]])
    for mode in ["topdown", "bottomup"]:
        row[f"delta_macro__{mode}"] = row[f"macro_f1__{mode}"] - row["macro_f1__original"]
        row[f"delta_jacc__{mode}"] = row[f"jaccard_acc__{mode}"] - row["jaccard_acc__original"]
        row[f"n_labels_improved__{mode}"] = int((f1[mode] > f1["original"] + 1e-9).sum())
        row[f"wilcoxon_p__{mode}"] = wilcoxon_signed_rank(f1[mode], f1["original"])["p"]
    return row

## Apply Post-Processing

Applies the post-processing to every complete configuration and counts how often it helps.

In [ ]:
all_runs = pd.concat([pd.read_csv(RES / RESULTS_CSV), load_extended()], ignore_index=True)
n_folds = all_runs[all_runs.strategy.isin(["br", "cc", "mo"])].groupby("config").fold.nunique()
postprocessing = pd.DataFrame(
    [
        postprocessing_row(config, all_runs[all_runs.config == config].iloc[0])
        for config in n_folds[n_folds == 5].index
    ]
)
postprocessing["wilcoxon_p_holm__topdown"] = holm_bonferroni(
    postprocessing["wilcoxon_p__topdown"].to_numpy()
)
save_table(postprocessing, "ext_postproc_hierarchy.csv", index=False)
print(
    "Configurations:",
    len(postprocessing),
    "| macro-F1 up (topdown):",
    int((postprocessing.delta_macro__topdown > 0).sum()),
    "| Jaccard up (topdown):",
    int((postprocessing.delta_jacc__topdown > 0).sum()),
    "| significant after Holm (topdown):",
    int((postprocessing.wilcoxon_p_holm__topdown < 0.05).sum()),
)
postprocessing.groupby(["algo", "strategy"])[
    ["delta_macro__topdown", "delta_jacc__topdown", "delta_macro__bottomup", "delta_jacc__bottomup"]
].mean().round(4)

## Top Configurations After Post-Processing

The two best configurations of each algorithm before and after topdown post-processing.

In [ ]:
shown = [
    "config",
    "macro_f1__original",
    "macro_f1__topdown",
    "jaccard_acc__original",
    "jaccard_acc__topdown",
    "subset_acc__original",
    "subset_acc__topdown",
    "pred_hierarchy_violation_rate__original",
    "n_labels_improved__topdown",
    "wilcoxon_p_holm__topdown",
]
postprocessing.sort_values("macro_f1__original", ascending=False).groupby("algo").head(2)[
    shown
].round(4)

## ML-ROS and Character Features Grid

ML-ROS clones minority instances up to 20 percent of the training data, close to the share added by MLSMOTE. If both ML-ROS and MLSMOTE fail to help, the conclusion that oversampling does not suit TF-IDF features on this data is stronger and does not depend on the MLSMOTE implementation. Character features and combined word and character features are tested because Ibrohim and Budi (2019) report character features among the best on this data. They are run only for LinearSVC and XGBoost, without imbalance treatment and with tuning.

In [ ]:
extended_grid = build_grid(EXTENDED_ALGOS_TO_RUN, ["br", "cc"], ["mlros"], tuned=(False, True))
extended_grid += build_grid(
    ["svm"] + (["xgb"] if RUN_GPU_MODELS else []),
    ["br", "cc"],
    ["none"],
    tuned=(True,),
    features=("char", "wordchar"),
)
if RUN_EXTENDED_GRID:
    run_grid(
        ROOT,
        extended_grid,
        n_jobs_cpu=N_JOBS_CPU,
        csv_name=EXTENDED_CSV,
        preds_subdir="preds_ext",
        verbose=False,
    )
extended = load_extended()
EXTENDED_ALGOS = [a for a in ["cnb", "svm", "xgb", "mlp"] if a in set(extended.algo)]
print("Rows in", EXTENDED_CSV, ":", len(extended))

## Classical Runs With Features

Combines the main grid, where every run uses word features, with the follow-up runs of the classical models, and keeps the configurations with all five folds.

In [ ]:
KEY_FEATURES = ["algo", "strategy", "condition", "features", "tuning"]
classical_runs = pd.concat(
    [
        pd.read_csv(RES / RESULTS_CSV).assign(features="word"),
        extended[extended.algo.isin(["cnb", "svm", "xgb", "mlp"])],
    ],
    ignore_index=True,
)
classical_runs["tuning"] = np.where(classical_runs.tuned.astype(bool), "tuned", "fixed")
n_folds = classical_runs.groupby(KEY_FEATURES).fold.nunique()
classical_runs = (
    classical_runs.set_index(KEY_FEATURES).loc[n_folds[n_folds == 5].index].reset_index()
)
per_label_features = classical_runs.groupby(KEY_FEATURES)[F1_COLUMNS].mean()
per_label_features.columns = LABELS
mean_features = classical_runs.groupby(KEY_FEATURES)[
    ["macro_f1", "minority_macro_f1", "jaccard_acc"]
].mean()

## Compare Configurations

Paired comparison of two classical configurations that may differ in features.

In [ ]:
def compare_configurations(key_a, key_b, family):
    comparison = paired_comparison(
        per_label_features.loc[key_a].to_numpy(),
        per_label_features.loc[key_b].to_numpy(),
        fold_macro_f1(classical_runs, KEY_FEATURES, key_a),
        fold_macro_f1(classical_runs, KEY_FEATURES, key_b),
        N_TRAIN,
        N_TEST,
    )
    return {
        "family": family,
        "algo": ALGO_NAMES[key_a[0]],
        "strategy": key_a[1].upper(),
        "A": "/".join(key_a[2:]),
        "B": "/".join(key_b[2:]),
        **comparison,
    }

## ML-ROS and Feature Comparisons

ML-ROS against the other imbalance treatments, and character features against word features.

In [ ]:
pairs = []
for algo in EXTENDED_ALGOS:
    for strategy in ["br", "cc"]:
        for tuning in ["fixed", "tuned"]:
            for a, b in [("none", "mlros"), ("mlsmote", "mlros"), ("weight", "mlros")]:
                pairs.append(
                    (
                        (algo, strategy, a, "word", tuning),
                        (algo, strategy, b, "word", tuning),
                        "ML-ROS",
                    )
                )
for algo in ["svm", "xgb"]:
    for strategy in ["br", "cc"]:
        for features in ["char", "wordchar"]:
            pairs.append(
                (
                    (algo, strategy, "none", "word", "tuned"),
                    (algo, strategy, "none", features, "tuned"),
                    "features",
                )
            )
comparison_rows = [
    compare_configurations(a, b, family)
    for a, b, family in pairs
    if a in per_label_features.index and b in per_label_features.index
]
extended_comparisons = pd.DataFrame(comparison_rows)
if len(extended_comparisons):
    extended_comparisons = add_holm_columns(extended_comparisons, ["wilcoxon_p"], by="family")
    extended_comparisons = move_columns_last(extended_comparisons, EFFECT_SIZE_COLUMNS)
save_table(extended_comparisons, "ext_stats_mlros_features.csv", index=False)

## Minority Precision and Recall

This table tests one explanation of why oversampling does not help, namely that the recall of the minority labels rises while their precision falls further. The values are means over five folds for Binary Relevance with word features, with and without tuning.

In [ ]:
minority_pr_rows = []
classical_indexed = classical_runs.set_index(KEY_FEATURES)
for algo in EXTENDED_ALGOS:
    for tuning in ["fixed", "tuned"]:
        for condition in ["none", "weight", "mlsmote", "mlros"]:
            key = (algo, "br", condition, "word", tuning)
            if key not in mean_features.index:
                continue
            runs = classical_indexed.loc[key]
            row = {"algo": ALGO_NAMES[algo], "tuning": tuning, "condition": condition}
            for label in MINORITY_LABELS:
                row[f"P_{label}"] = runs[f"p__{label}"].mean()
                row[f"R_{label}"] = runs[f"r__{label}"].mean()
                row[f"F1_{label}"] = runs[f"f1__{label}"].mean()
            row["P_mean"] = np.mean([row[f"P_{label}"] for label in MINORITY_LABELS])
            row["R_mean"] = np.mean([row[f"R_{label}"] for label in MINORITY_LABELS])
            minority_pr_rows.append(row)
minority_pr = pd.DataFrame(minority_pr_rows)
save_table(minority_pr, "ext_minority_precision_recall.csv", index=False, decimals=3)
minority_pr[
    ["algo", "tuning", "condition", "P_mean", "R_mean"]
    + [f"F1_{label}" for label in MINORITY_LABELS]
].round(3)

## Revised Transformer Protocol

Every configuration in TRANSFORMER_CONFIGS is trained with the revised protocol on the five folds, and the ensembles only use stored probabilities. The final configuration is chosen by inner validation macro F1, not by the test score, and its test score is reported as it is. Some choices come from another team's experiments on the same dataset, an IndoBERT model with macro F1 0.759 under a nested protocol, and from the IndoBERTweet preprocessing of Koto et al. (2021).

In [ ]:
if RUN_TRANSFORMERS:
    run_transformer_configs(ROOT, TRANSFORMER_CONFIGS)
    for ensemble_name, members in TRANSFORMER_ENSEMBLES.items():
        ensemble_transformers(ROOT, members, ensemble_name)
else:
    print(
        "Revised transformer protocol skipped (no GPU or RUN_TRANSFORMERS=False). "
        "Stored results are used."
    )
extended = load_extended()

## Transformer Summary Table

Every transformer configuration and ensemble, with the baseline protocol runs added for reference.

In [ ]:
transformer_runs = (
    extended[extended.algo.isin(["tf", "ens"])].copy() if len(extended) else pd.DataFrame()
)
if len(transformer_runs):
    n_folds = transformer_runs.groupby("config").fold.nunique()
    transformer_runs = transformer_runs[transformer_runs.config.isin(n_folds[n_folds == 5].index)]
    columns = [
        "macro_f1",
        "micro_f1",
        "minority_macro_f1",
        "jaccard_acc",
        "inner_macro_f1",
        "train_time_s",
    ]
    transformer_table = transformer_runs.groupby("config")[columns].agg(["mean", "std"])
    transformer_table.columns = [f"{a}_{b}" for a, b in transformer_table.columns]
    if "best_epoch" in transformer_runs:
        transformer_table["best_epochs"] = transformer_runs.groupby("config").best_epoch.apply(
            lambda s: "/".join(map(str, s.astype("Int64")))
        )
    baseline_runs = pd.read_csv(RES / RESULTS_CSV).query("algo == 'bert'")
    for config, runs in baseline_runs.groupby("config", sort=False):
        transformer_table.loc[
            f"baseline protocol: {config}",
            [
                "macro_f1_mean",
                "macro_f1_std",
                "minority_macro_f1_mean",
                "jaccard_acc_mean",
                "train_time_s_mean",
            ],
        ] = [
            runs.macro_f1.mean(),
            runs.macro_f1.std(),
            runs.minority_macro_f1.mean(),
            runs.jaccard_acc.mean(),
            runs.train_time_s.mean(),
        ]
    transformer_table = transformer_table.sort_values("macro_f1_mean", ascending=False)
    save_table(transformer_table, "ext_transformer_table.csv")
    tuned_runs = transformer_runs[transformer_runs.config.str.endswith("__tuned")]
    picked = tuned_runs[
        tuned_runs.config == tuned_runs.groupby("config").inner_macro_f1.mean().idxmax()
    ]
    print(
        "Best by inner validation macro F1 with tuned thresholds:",
        picked.config.iloc[0],
        f"| inner {picked.inner_macro_f1.mean():.4f} | test {picked.macro_f1.mean():.4f} "
        f"± {picked.macro_f1.std():.4f}",
    )
    display(transformer_table.round(4))
else:
    print("No results from the revised transformer protocol yet.")

## Threshold Strategy and Final Model

The configuration and threshold strategy with the highest inner validation estimate become the final model. There is no retraining.

In [ ]:
if len(transformer_runs):
    threshold_rows, threshold_store = [], {}
    base_configs = sorted({c.removesuffix("__tuned") for c in transformer_runs.config.unique()})
    for base in base_configs:
        for k in range(5):
            evaluation = evaluate_threshold_strategies(
                Y_BY_ID, RES / "preds_ext" / f"{base}__f{k}.npz", seed=k
            )
            for strategy, values in evaluation.items():
                threshold_rows.append(
                    {
                        "config": base,
                        "fold": k,
                        "strategy": strategy,
                        "inner_2fold": values["inner_2fold"],
                        "test_macro_f1": values["test_macro_f1"],
                    }
                )
                threshold_store[(base, strategy, k)] = values
    threshold_results = pd.DataFrame(threshold_rows)
    threshold_table = (
        threshold_results.groupby(["config", "strategy"])[["inner_2fold", "test_macro_f1"]]
        .mean()
        .unstack("strategy")
    )
    display(save_table(threshold_table, "ext_threshold_strategies.csv"))
    inner_estimate = threshold_results.groupby(["config", "strategy"]).inner_2fold.mean()
    FINAL_CONFIG, FINAL_STRATEGY = inner_estimate.idxmax()
    print(
        f"Final model (chosen on inner validation data): {FINAL_CONFIG} "
        f"with thresholds '{FINAL_STRATEGY}' | inner 2-fold {inner_estimate.max():.4f}"
    )
    final_rows = []
    for k in range(5):
        values = threshold_store[(FINAL_CONFIG, FINAL_STRATEGY, k)]
        y_test = Y_BY_ID.loc[values["test_ids"]].to_numpy(int)
        y_pred = (values["test_prob"] > values["thresholds"][None, :]).astype(int)
        final_rows.append({"fold": k, **evaluate_predictions(y_test, y_pred)})
    final_model_folds = pd.DataFrame(final_rows)
    save_table(final_model_folds, "ext_final_model_folds.csv", index=False)
    fm = final_model_folds.mean()
    print(
        f"Final model test scores: macro-F1 {fm.macro_f1:.4f} "
        f"± {final_model_folds.macro_f1.std():.4f} | "
        f"minority macro-F1 {fm.minority_macro_f1:.4f} | Jaccard {fm.jaccard_acc:.4f} | "
        f"micro-F1 {fm.micro_f1:.4f} | Hamming accuracy {1 - fm.hamming_loss:.4f}"
    )

## Per-Label F1 and Fold Scores

Returns the mean F1 of each label and the per-fold macro F1 of one configuration.

In [ ]:
def per_label_and_folds(runs, config):
    d = runs[runs.config == config]
    return (
        np.array([d[f"f1__{label}"].mean() for label in LABELS]),
        d.sort_values("fold").macro_f1.to_numpy(),
    )

## Final Model Against Earlier Models

The F1 of each label for the final model, IndoBERTweet with the baseline protocol and the best LinearSVC configurations, followed by paired tests of the final model against each of them.

In [ ]:
if len(transformer_runs):
    all_runs = pd.concat([pd.read_csv(RES / RESULTS_CSV), extended], ignore_index=True)
    models = {
        "IndoBERTweet (baseline protocol)": per_label_and_folds(
            all_runs, "bert__mo__weight__tuned"
        ),
        "LinearSVC BR tuned (word)": per_label_and_folds(all_runs, "svm__br__none__tuned"),
        "LinearSVC CC tuned (word + char)": per_label_and_folds(
            all_runs, "svm__cc__none__wordchar__tuned"
        ),
        "Final model": (
            np.array([final_model_folds[f"f1__{label}"].mean() for label in LABELS]),
            final_model_folds.sort_values("fold").macro_f1.to_numpy(),
        ),
    }
    models = {name: values for name, values in models.items() if len(values[1]) == 5}
    final_per_label = pd.DataFrame(
        {name: values[0] for name, values in models.items()}, index=LABELS
    )
    final_per_label.loc["macro-F1"] = final_per_label.mean()
    display(save_table(final_per_label, "ext_f1_per_label_final.csv", decimals=3))
    f1_final, folds_final = models["Final model"]
    comparator_rows = []
    for name, (f1_other, folds_other) in models.items():
        if name == "Final model":
            continue
        wilcoxon = wilcoxon_signed_rank(f1_final, f1_other)
        ttest = corrected_resampled_ttest(folds_final, folds_other, N_TRAIN, N_TEST)
        comparator_rows.append(
            {
                "comparator": name,
                "delta_macro_f1": f1_final.mean() - f1_other.mean(),
                "n_labels_better": int((f1_final > f1_other).sum()),
                "wilcoxon_p": wilcoxon["p"],
                "corrected_t_p": ttest["p"],
                "wilcoxon_stat": wilcoxon["stat"],
                "rank_biserial": wilcoxon["rank_biserial"],
                "corrected_t": ttest["t"],
            }
        )
    final_comparison = pd.DataFrame(comparator_rows)
    if len(final_comparison):
        final_comparison["wilcoxon_p_holm"] = holm_bonferroni(
            final_comparison.wilcoxon_p.to_numpy()
        )
        final_comparison = move_columns_last(final_comparison, EFFECT_SIZE_COLUMNS)
    display(save_table(final_comparison, "ext_stats_final_model.csv", index=False))

# Cross-Corpus Evaluation

The models are trained on all the data of Ibrohim and Budi (2019) and tested on the latest IndoToxic2024 release, IndoDiscourse version 2 by Susanto et al. (2024). It is distributed under the Apache 2.0 licence and downloaded from the HuggingFace dataset Exqrch/IndoDiscourse.

The label mapping is an assumption. HS is mapped to toxicity, which the IndoToxic2024 paper uses as its hate speech task, and to identity_attack, a narrower comparison. Abusive is mapped to profanity_obscenity.

The models are the two best classical models and Complement Naive Bayes, with hyperparameters and thresholds from 15 percent inner validation data, plus the revised protocol transformers, the final model and its comparison ensemble with threshold 0.5.

This is a cross-corpus generalisation test, not a pure temporal test. The 2024 corpus differs in year, platform, annotation guideline and label definitions. Its platforms are Facebook, Instagram, X and fact-checking articles. The drop in scores therefore cannot be attributed to one factor.

## Download and Score

Downloads the corpus if it is missing and scores every model on it.

In [ ]:
if not CROSS_CORPUS_FILE.exists():
    try:  # automatic download, needs internet access
        from huggingface_hub import hf_hub_download

        CROSS_CORPUS_FILE.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(
            hf_hub_download(
                "Exqrch/IndoDiscourse",
                "indotoxic2024_annotated_data_v2_final.csv",
                repo_type="dataset",
            ),
            CROSS_CORPUS_FILE,
        )
    except Exception as e:
        print("Could not download the 2024 corpus:", repr(e))
if RUN_CROSS_CORPUS is None:
    RUN_CROSS_CORPUS = CROSS_CORPUS_FILE.exists()
if RUN_CROSS_CORPUS:
    corpus_2024, cross_corpus, oov_summary, oov_top = run_cross_corpus(
        ROOT, run_transformers=RUN_TRANSFORMERS
    )
    print(f"2024 corpus: {len(corpus_2024)} texts after aggregation")
else:
    print("The 2024 corpus is not available; this part is skipped.")

## In-Domain Comparison

Adds the in-domain F1 of the same label from cross-validation on Ibrohim and Budi, and the drop from in-domain to cross-corpus.

In [ ]:
IN_DOMAIN_CONFIG = {
    "svm__br__none__word__tuned": "svm__br__none__tuned",
    "svm__cc__none__wordchar__tuned": "svm__cc__none__wordchar__tuned",
    "cnb__br__weight__word__tuned": "cnb__br__weight__tuned",
    **{f"tf__{name} (threshold 0.5)": f"tf__{name}" for name in CROSS_TRANSFORMERS},
    "ensemble IndoBERTweet + IndoBERT-p1 (6 epochs)": "ens__indobertweet+indobert_p1",
    FINAL_ENSEMBLE_NAME: "ens__e10_indobertweet_x3+indobert_p1",
}


def in_domain_f1(runs, model_name, label):
    model_runs = runs[runs.config == IN_DOMAIN_CONFIG.get(model_name)]
    return model_runs[f"f1__{label}"].mean() if len(model_runs) else np.nan


if RUN_CROSS_CORPUS:
    all_runs = pd.concat([pd.read_csv(RES / RESULTS_CSV), load_extended()], ignore_index=True)
    cross_corpus["f1_in_domain_cv"] = [
        in_domain_f1(all_runs, m, label)
        for m, label in zip(cross_corpus.model, cross_corpus.source_label)
    ]
    cross_corpus["f1_drop"] = cross_corpus.f1_in_domain_cv - cross_corpus.f1_pos
    save_table(cross_corpus, "crosscorpus_results.csv", index=False)
    print(
        f"OOV tokens: {oov_summary['oov_token_rate'] * 100:.1f}% | "
        f"OOV word types: {oov_summary['oov_type_rate'] * 100:.1f}%"
    )
    display(cross_corpus.round(3))

## Cross-Corpus Chart

The F1 of the positive class on IndoToxic2024 for each model and label, followed by the most frequent words of the 2024 corpus that never occur in the training data.

In [ ]:
if RUN_CROSS_CORPUS:
    pivot = cross_corpus.pivot_table(index="model", columns="target_label", values="f1_pos")
    fig, ax = plt.subplots(figsize=(8, 5.5))
    pivot.plot.barh(ax=ax, color=["#8e44ad", "#c0392b", "#2e86c1"])
    ax.set_xlabel("F1 of the positive class on IndoToxic2024")
    ax.set_ylabel("")
    ax.legend(title="IndoToxic2024 label", loc="lower right", fontsize=8)
    fig.savefig(FIG / "crosscorpus_f1.png")
    plt.show()
    display(oov_top.head(15).T)